# CS6140 Machine Learning — Fall 2026
# HW6 Starter — Convolutional and Recurrent Networks

This notebook implements `HW6_26F.html`. Read the HTML for the full problem statements; the markdown
cells here repeat the key instructions next to the code.

| Problem | Topic | Points |
|---|---|---|
| 1 | A CNN from its building blocks (Fashion-MNIST) | 50 |
| 2 | Deeper CNNs on natural images: residual blocks, augmentation, transfer (CIFAR-10) | 50 |
| 3 | A recurrent language model (TinyStories) | 60 |
| 4 | Sequence-to-sequence translation, Spanish → English | 50 |
| 5, 6 | THEORY: why an LSTM reaches further back; what convolution is robust to | 25 + 25 |
| 8, 11 | optional: an LSTM cell from scratch; why dev mode looks the way it does (7, 9, 10 are HTML-only) | no credit |

**How to work through this notebook.** Everything runs as given except the lines marked
`### TODO_STUDENT ###`: their code has been replaced by `...` plus a one-line description of what
goes there. Replace each `...` with real code. Until you do, the cell (or a later cell that uses it)
will fail or print nonsense. That's expected, not a bug. Written answers go in the markdown cells
that say *Your answer here*; each has a short *How to think about it* hint.

**Checkpoint cells** follow each piece you implement. Most copy your weights into the matching PyTorch
module (or call the matching PyTorch function) and check that both give the same numbers, in float64
on the CPU. Don't move on until the checkpoint above you passes: a network built on a wrong piece can
still *look* like it's learning.

**Work top to bottom.** Problem 1(B) builds LeNet from your Problem 1(A) layers; Problem 1(D)'s batch
norm goes into a LeNet; Problem 3's next-word setup is what Problem 4's decoder does; Problem 4's last
cell saves `results_hw6.json` and `hw6_seq2seq.pt`, which HW7 reads.

**Two modes: `MODE = "dev"` and `MODE = "full"`** (one switch, in the *Run settings* cell; nothing else
to choose — both settings are given and printed by that cell).
* **dev** (the default): small data subsets and the same networks at smaller sizes. Every training cell
  takes under about a minute on a laptop CPU. Use it to **write and debug** your code: the checkpoints
  and the "overfit one batch" checks are as trustworthy as in full mode. Every dev result is labelled
  **[DEV — not for answers]**: dev accuracies, perplexities and BLEU scores are meaningless.
* **full**: the real data and models. **Every written answer, and anything you show in office hours,
  must come from a full run.** Each training run saves its model and history (folder `hw6_runs/`); when
  you re-run a cell, it reloads the saved run and says so ("loaded saved full run from …; set
  RETRAIN=True or delete it to retrain"), so a finished problem is never retrained by accident.
  Dev always retrains and never saves into `hw6_runs/`.
* **HW7 needs your own HW6 full results**: Problem 4's last cell writes `results_hw6.json` (with Problem
  3's language models) and `hw6_seq2seq.pt`. Keep them. (Dev writes `results_hw6_dev.json` and
  `hw6_seq2seq_dev.pt`, which HW7 does not accept.)

**Expected full-run time.** Measured on an Apple M1 Max (32-core GPU): about 25 minutes for the whole
notebook, the longest single run about 5 minutes, plus the first-run downloads. *Estimates* for other
machines: base M1/M2/M3 laptops (8–10-core GPU) about 2.5–3.5× longer; Colab's free T4 GPU about the same
as the M1 Max; CPU only: use Colab, or run full mode overnight. A dev run of the whole notebook takes about
4 minutes on an M1 Max CPU (estimate: under 10 minutes on an average laptop CPU). Next to each comparison, an **Expected in a full run** cell gives the range your full
numbers should land in.

**Setup.** If the first cell fails, see the course's *NN setup page*. Keep this folder as you downloaded it
(this notebook, `lm_common.py`, `seq2seq_common.py`). The data goes into the shared `data/` folder next to
your HW folders and is downloaded on first run (about 260 MB: CIFAR-10, Fashion-MNIST, TinyStories, the
Spanish–English pairs, the pretrained ResNet-18 weights); nothing to do by hand.

**Requirements.**
1. Fill in every `TODO_STUDENT` blank in the required problems according to the algorithm steps
   covered in lecture and the linked materials, and make sure the notebook runs top to bottom
   without errors. (Blanks in the optional problems are not required.)
2. Understand the *whole* notebook — including the provided/given code, not just the blanks you
   filled in — well enough to explain any part of it during office hours.


In [ ]:
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
import torchvision
from d2l import torch as d2l
from tqdm import tqdm
from IPython.display import display


def get_device():
    if torch.cuda.is_available():            # NVIDIA GPU
        return torch.device('cuda')
    if torch.backends.mps.is_available():    # Apple Silicon GPU
        return torch.device('mps')
    return torch.device('cpu')


DEVICE = get_device()
if DEVICE.type == 'mps':   # d2l only knows CUDA: point its GPU helpers at Apple's GPU
    d2l.num_gpus = lambda: 1
    d2l.gpu = lambda i=0: DEVICE
    d2l.try_gpu = lambda i=0: DEVICE
print('Best available device:', DEVICE)

# HW6's networks are big enough that the GPU pays off: everything trains on DEVICE.
# (The checkpoint cells run on the CPU in float64, where "same output" can be checked to 1e-10.)
TRAIN_DEVICE = DEVICE

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import os
import urllib.request

# The shared data folder: cs6140/data, next to your HW folders (NN setup page, section 3).
DATA = next((d for d in ["../data", "../../data", "../../../data"] if os.path.isdir(d)), "../data")
COURSE_DATA_URL = "https://www.khoury.northeastern.edu/home/vip/teach/MLcourse/data"


def course_file(name, fallback_url=None):
    """DATA/name. Downloaded the first time from the course website (or from fallback_url)."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        for url in [f"{COURSE_DATA_URL}/{name}"] + ([fallback_url] if fallback_url else []):
            try:
                print(f"downloading {name} from {url} ...")
                urllib.request.urlretrieve(url, path + ".part")
                os.replace(path + ".part", path)
                break
            except OSError as err:
                print(f"  failed: {err}")
        else:
            raise FileNotFoundError(f"could not download {name}: put it into {os.path.abspath(DATA)}")
    return path
TORCH_DATA = f"{DATA}/torchvision"           # Fashion-MNIST + CIFAR-10 download here (~260 MB)


# --- Progress bars (nothing to do here) ---
class _BarOutput:
    """Where tqdm writes its bar: ONE notebook output, updated in place (not a new line per redraw).
    Inside a d2l live plot, the bar becomes the plot's title instead, because d2l clears the cell's
    output on every redraw and would erase a separate bar."""

    def __init__(self, board=None):
        self.board, self.handle = board, None

    def write(self, text):
        text = text.strip("\r\n")
        if not text:
            return
        if self.board is not None:
            if self.board.fig is not None:
                self.board.fig.suptitle(text, fontsize=8, family="monospace")
        elif self.handle is None:
            self.handle = display({"text/plain": text}, raw=True, display_id=True)
        else:
            self.handle.update({"text/plain": text}, raw=True)

    def flush(self):
        pass


def progress(total, desc, unit="batch", board=None):
    """A progress bar for a training loop: call .update(1) per step, .close() at the end."""
    width = 25 if board is None else 12
    fmt = "{desc}: {percentage:3.0f}%|{bar:" + str(width) + "}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]"
    return tqdm(total=total, desc=desc, unit=unit, file=_BarOutput(board), mininterval=0.5, ascii=False,
                bar_format=fmt)


def _fit_with_progress(self, model, data):
    """d2l.Trainer.fit, plus a progress bar over all training batches (shown as the live plot's title)."""
    self.prepare_data(data)
    self.prepare_model(model)
    self.optim = model.configure_optimizers()
    self.epoch = self.train_batch_idx = self.val_batch_idx = 0
    board = model.board if model.board.display else None
    if board is not None:
        board.figsize = (5.5, 3.2)                  # room for the bar in the title
    bar = progress(self.max_epochs * self.num_train_batches, type(model).__name__, board=board)
    loader = self.train_dataloader
    self.train_dataloader = _TickingLoader(loader, bar)
    try:
        for self.epoch in range(self.max_epochs):
            self.fit_epoch()
            bar.set_postfix(epoch=self.epoch + 1)
    finally:
        self.train_dataloader = loader
        bar.close()


class _TickingLoader:
    """Wraps a DataLoader so that every batch it hands out advances the progress bar."""
    def __init__(self, loader, bar):
        self.loader, self.bar = loader, bar
    def __iter__(self):
        for batch in self.loader:
            yield batch
            self.bar.update(1)
    def __len__(self):
        return len(self.loader)


d2l.Trainer.fit = _fit_with_progress


## Run settings and shared helpers (nothing to do here)

* **`MODE`**: `"dev"` (small data, small models: write and debug; results labelled *not for answers*) or
  `"full"` (the real runs, for every answer). The cell prints both settings and marks the active one.
  `CONFIG` is given: don't change the sizes.
* **`RETRAIN`**: in full mode, saved runs in `hw6_runs/` are reloaded instead of retrained, unless this is
  `True` (or you delete the file).
* **`RUN_OPTIONAL`** runs optional Problem 8.
* `seq2seq_common.py` and `lm_common.py` (next to this notebook) hold the data and training code shared with HW7,
  so both homeworks see exactly the same data. Read them: they are part of the "given code" you should
  understand. Keep them next to this notebook.
* `train_classifier` is the image-classification training loop for Problems 1–2 (Adam, minibatches,
  accuracy after every epoch). `check` is used by every checkpoint cell.


In [ ]:
import os
import math
import collections

_places = [".", "../../code_common"]          # next to this notebook, or the instructor copies (2.5_NN/code_common)
COMMON = next((p for p in _places if os.path.exists(f"{p}/seq2seq_common.py")), None)
if COMMON is None:
    raise FileNotFoundError("seq2seq_common.py and lm_common.py not found in any of: "
                            + ", ".join(os.path.abspath(p) for p in _places)
                            + ". Put both files next to this notebook.")
course_file("sentence_pairs_large.tsv")      # the Spanish-English pairs (Problem 4), 24 MB
sys.path.insert(0, COMMON)
import lm_common
import seq2seq_common

MODE = "dev"          # "dev": write and debug (CPU, minutes)  |  "full": the real runs, for every answer
RETRAIN = False        # full mode: True retrains even when a saved run exists in hw6_runs/
RUN_OPTIONAL = False
torch.hub.set_dir(f"{TORCH_DATA}/hub")      # where torchvision keeps the pretrained ResNet-18 weights

# Both settings are given; students never pick sizes. dev = small subsets, the same networks at smaller sizes.
CONFIG = {
    "full": dict(fm_train=60000, fm_test=10000, fm_epochs=10,
                 cifar_train=50000, cifar_test=10000, cifar_epochs=10, resnet_widths=(32, 64, 128),
                 sub_per_class=500, sub_epochs=30, probe_epochs=100, ft_epochs=5, resize=128,
                 lm_batch=64, lm_len=64, lm_min_freq=10, lm_embed=256, lm_hidden=512, lm_steps=3000,
                 mt_train=100_000, mt_embed=256, mt_hidden=512, mt_layers=2, mt_steps=2500,
                 est="M1 Max GPU: 0.5-4 min per training cell, ~25 min in total"),
    "dev": dict(fm_train=6000, fm_test=2000, fm_epochs=2,
                cifar_train=2000, cifar_test=1000, cifar_epochs=3, resnet_widths=(16, 32, 64),
                sub_per_class=50, sub_epochs=5, probe_epochs=30, ft_epochs=1, resize=64,
                lm_batch=32, lm_len=32, lm_min_freq=50, lm_embed=64, lm_hidden=128, lm_steps=300,
                mt_train=10_000, mt_embed=64, mt_hidden=128, mt_layers=2, mt_steps=800,
                est="M1 Max CPU: under 45 s per training cell, ~4 min in total"),
}
assert MODE in CONFIG, 'MODE must be "dev" or "full"'
C = CONFIG[MODE]
TAG = "[DEV — not for answers] " if MODE == "dev" else ""     # printed in front of every dev result
RUNS = "hw6_runs"                                             # saved full runs (models + histories)
print(f"{'setting':15s} {'dev':>22s} {'full':>22s}")
for k in CONFIG["full"]:
    if k != "est":
        print(f"{k:15s} {str(CONFIG['dev'][k]):>22s} {str(CONFIG['full'][k]):>22s}")
print(f"MODE = {MODE!r} ({C['est']}) | RETRAIN = {RETRAIN} | RUN_OPTIONAL = {RUN_OPTIONAL}")
if MODE == "dev":
    print(TAG + "dev mode: write and debug here; every answer must come from a FULL run.")


def cached(key, model, train):
    """Full mode: reload a saved run (model + history) if there is one, else train() and save it.
    Dev mode: always train(), never save. Returns the history dict."""
    path = f"{RUNS}/{key}.pt"
    if MODE == "full" and not RETRAIN and os.path.exists(path):
        saved = torch.load(path, map_location="cpu", weights_only=False)
        model.load_state_dict(saved["state_dict"])
        model.to(TRAIN_DEVICE)
        print(f"loaded saved full run from {path}; set RETRAIN=True or delete it to retrain")
        return saved["hist"]
    hist = train()
    if MODE == "full":
        os.makedirs(RUNS, exist_ok=True)
        torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "hist": hist}, path)
    return hist


def overfit_one_batch(name, model, batch_loss, clip_fn=None, lr=1e-2, target=0.1, max_seconds=30):
    """Given check: train on ONE small batch until the loss is near 0. A model that can't memorize a
    handful of examples has a bug (wrong wiring, a detached tensor, a mask that hides everything...)."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    t0, step, first = time.time(), 0, None
    while True:
        loss = batch_loss(model)
        first = loss.item() if first is None else first
        opt.zero_grad()
        loss.backward()
        if clip_fn is not None:
            clip_fn(list(model.parameters()), 1.0)
        opt.step()
        step += 1
        if loss.item() < target or time.time() - t0 > max_seconds:
            break
    assert loss.item() < target, (f"{name}: one batch not memorized in {max_seconds} s (loss {first:.2f} -> "
                                  f"{loss.item():.2f}): look for a bug before training for real")
    print(f"CHECK passed: {name} memorizes one batch (loss {first:.2f} -> {loss.item():.3f} in {step} steps, "
          f"{time.time() - t0:.1f} s)")


def n_params(model):
    return sum(p.numel() for p in model.parameters())


def check(name, mine, ref, tol=1e-8):
    """Checkpoint: the max absolute difference between two tensors must be below tol."""
    mine, ref = torch.as_tensor(mine, dtype=torch.float64), torch.as_tensor(ref, dtype=torch.float64)
    assert mine.shape == ref.shape, f"{name}: shape {tuple(mine.shape)}, expected {tuple(ref.shape)}"
    gap = (mine - ref).abs().max().item() if mine.numel() else 0.0
    assert gap < tol, f"{name}: differs from the reference by {gap:.3g} (tolerance {tol:g})"
    print(f"CHECK passed: {name} (max difference {gap:.1e})")


@torch.no_grad()
def accuracy(model, X, y, batch_size=1000, transform=None):
    """Fraction of correct predictions, in evaluation mode (batch norm uses its running averages)."""
    was_training = model.training
    model.eval()
    correct = 0
    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size]
        if transform is not None:
            xb = transform(xb)
        correct += (model(xb).argmax(1) == y[i:i + batch_size]).sum().item()
    model.train(was_training)
    return correct / len(X)


def train_classifier(model, X, y, X_test, y_test, epochs, lr, batch_size=256, augment=None, transform=None,
                     onecycle=False, weight_decay=0.0, params=None, desc="model", seed=SEED):
    """Adam(W) on shuffled minibatches; returns per-epoch train loss, train/test accuracy, and seconds.
    augment: applied to each TRAINING batch only.  transform: applied to every batch (train and test).
    onecycle: learning rate warms up, then decays to ~0 (used for CIFAR-10)."""
    model.to(X.device).train()
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr,
                            weight_decay=weight_decay)
    n_batches = (len(X) + batch_size - 1) // batch_size
    sched = (torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * n_batches)
             if onecycle else None)
    g = torch.Generator().manual_seed(seed)
    hist = {"train_loss": [], "train_acc": [], "test_acc": []}
    bar = progress(epochs * n_batches, desc)
    t0 = time.time()
    for epoch in range(epochs):
        perm = torch.randperm(len(X), generator=g).to(X.device)
        total = 0.0
        for i in range(0, len(X), batch_size):
            idx = perm[i:i + batch_size]
            xb = X[idx]
            if augment is not None:
                xb = augment(xb)
            if transform is not None:
                xb = transform(xb)
            loss = F.cross_entropy(model(xb), y[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            if sched is not None:
                sched.step()
            total += loss.item() * len(idx)
            bar.update(1)
        hist["train_loss"].append(total / len(X))
        hist["train_acc"].append(accuracy(model, X[:10000], y[:10000], transform=transform))
        hist["test_acc"].append(accuracy(model, X_test, y_test, transform=transform))
        bar.set_postfix(epoch=epoch + 1, test_acc=f"{hist['test_acc'][-1]:.3f}")
    bar.close()
    hist["seconds"] = time.time() - t0
    return hist


def plot_histories(histories, key="test_acc", title="", ax=None, ylabel="test accuracy"):
    ax = ax or plt.subplots(figsize=(5.5, 3.5))[1]
    for name, h in histories.items():
        ax.plot(range(1, len(h[key]) + 1), h[key], marker="o", ms=3, label=name)
    ax.set_xlabel("epoch"); ax.set_ylabel(ylabel); ax.set_title(TAG + title); ax.grid(alpha=0.3); ax.legend(fontsize=8)
    return ax


SUMMARY = {}       # key numbers, printed together at the end of the notebook


---
# PROBLEM 1 — A CNN from Its Building Blocks (Fashion-MNIST) [50 points]

HW5's MLP flattened each 28×28 image into 784 numbers, so it had to learn separately that a sleeve at
the left and a sleeve at the right look alike. A convolution slides **one small kernel** over the whole
image: the same few weights are used at every position (*weight sharing*), and each output looks only at
a small patch (*locality*). Notation as in class (and d2l 7.1–7.3): input $X$, kernel $K$ of size
$k_h\times k_w$, output $Y_{i,j}=\sum_{a}\sum_{b}K_{a,b}\,X_{i+a,\,j+b}$ (*cross-correlation*, which deep
learning calls convolution).

*Library calls:* `torch.nn.functional.conv2d`, `torch.nn.functional.max_pool2d`, `torch.nn.Conv2d`,
`torch.nn.BatchNorm2d`.

**Data.** Fashion-MNIST, the same images as HW5 Problem 2, as tensors of shape (N, 1, 28, 28) with
pixels in [0, 1], kept on the GPU (the dataset is small enough).


In [ ]:
def load_fashion(train):
    ds = torchvision.datasets.FashionMNIST(root=TORCH_DATA, train=train, download=True)
    return (ds.data.float() / 255).unsqueeze(1).to(TRAIN_DEVICE), ds.targets.to(TRAIN_DEVICE)


X_fm, y_fm = load_fashion(train=True)
X_fm_test, y_fm_test = load_fashion(train=False)
X_fm, y_fm = X_fm[:C["fm_train"]], y_fm[:C["fm_train"]]                       # dev: a subset
X_fm_test, y_fm_test = X_fm_test[:C["fm_test"]], y_fm_test[:C["fm_test"]]
FM_CLASSES = torchvision.datasets.FashionMNIST.classes
print("Fashion-MNIST:", tuple(X_fm.shape), tuple(X_fm_test.shape))

EPOCHS_FM = C["fm_epochs"]             # Problem 1's training runs (all of them, your LeNet included)


## (A) Convolution and pooling, from scratch [15 points]

Four steps, each checked against hand-computed numbers (from lecture) and against PyTorch:

1. `corr2d(X, K)`: one channel, two loops over the output positions (d2l 7.2).
2. Several channels (d2l 7.4): with $c_i$ input channels, the kernel has one $k_h\times k_w$ slice per
   input channel and the results are **summed**; with $c_o$ output channels there is one such kernel per
   output channel, and the results are **stacked**.
3. **Convolution as one matrix product.** Copy every $k_h\times k_w\times c_i$ patch of the input into
   one column of a matrix (*unfold*, also called *im2col*); then all outputs at all positions are one
   product (kernel matrix) × (patch matrix). This is one way libraries implement convolution, and it is
   fast because a matrix product is what GPUs do best. `unfold` builds the patch matrix with one shifted
   copy of $X$ per kernel offset $(a, b)$ — only $k_h k_w$ slices, no loop over positions. Example: a
   3×3 image and a 2×2 kernel give a 2×2 output ($h' = w' = 2$). Kernel weight $K_{0,1}$ (offset
   $(a, b) = (0, 1)$) multiplies, at output position $(i, j)$, the pixel $X_{i,\,j+1}$; over all four
   positions these pixels form the slice `X[:, :, 0:2, 1:3]` of shape $(n, c, 2, 2)$. One such slice per
   offset, stacked, is the patch matrix.
4. Pooling: max or average over each window (d2l 7.5).

Weight layout: PyTorch stores a convolution's weight as $(c_o, c_i, k_h, k_w)$.


In [ ]:
def corr2d(X, K):
    """2-D cross-correlation of one channel X (h x w) with kernel K (kh x kw), with loops."""
    kh, kw = K.shape
    Y = torch.zeros((X.shape[0] - kh + 1, X.shape[1] - kw + 1), dtype=X.dtype)
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = ...  ### TODO_STUDENT ### the kh x kw window of X at (i, j), times K, summed
    return Y


X = torch.arange(9.0).reshape(3, 3)
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
print(corr2d(X, K))
check("corr2d on the lecture example", corr2d(X, K), [[19, 25], [37, 43]])
Xr, Kr = torch.randn(7, 9, dtype=torch.float64), torch.randn(3, 2, dtype=torch.float64)
check("corr2d vs F.conv2d", corr2d(Xr, Kr), F.conv2d(Xr[None, None], Kr[None, None])[0, 0])


In [ ]:
def corr2d_multi_in(X, K):
    """X: (c_i, h, w), K: (c_i, kh, kw) -> (h', w'): one corr2d per input channel, summed."""
    return ...  ### TODO_STUDENT ### corr2d of each channel with its kernel slice, summed over channels


def corr2d_multi_in_out(X, K):
    """X: (c_i, h, w), K: (c_o, c_i, kh, kw) -> (c_o, h', w'): one multi-input result per output channel."""
    return ...  ### TODO_STUDENT ### one corr2d_multi_in per output channel, stacked


X2 = torch.stack([torch.arange(9.0).reshape(3, 3), torch.arange(1.0, 10.0).reshape(3, 3)])
K2 = torch.stack([torch.tensor([[0.0, 1.0], [2.0, 3.0]]), torch.tensor([[1.0, 2.0], [3.0, 4.0]])])
check("2 input channels, lecture example", corr2d_multi_in(X2, K2), [[56, 72], [104, 120]])
Xr, Kr = torch.randn(3, 6, 5, dtype=torch.float64), torch.randn(4, 3, 2, 3, dtype=torch.float64)
check("multi-in/multi-out vs F.conv2d", corr2d_multi_in_out(Xr, Kr), F.conv2d(Xr[None], Kr)[0])


In [ ]:
def unfold(X, kh, kw):
    """Patch matrix. X: (n, c_i, h, w) -> (n, c_i*kh*kw, h'*w'): column p holds the patch at output position p."""
    n, c, h, w = X.shape
    ho, wo = h - kh + 1, w - kw + 1
    # e.g. offset (a, b) = (0, 1) is the slice X[:, :, 0:ho, 1:1 + wo], of shape (n, c, ho, wo)
    shifted = [X[:, :, a:a + ho, b:b + wo] for a in range(kh) for b in range(kw)]  # (given) one (n, c, h', w') slice of X per kernel offset (a, b), row-major over (a, b)
    return torch.stack(shifted, dim=2).reshape(n, c * kh * kw, ho * wo)


def conv2d_unfold(X, W, b=None, padding=0):
    """Convolution as one matrix product. X: (n, c_i, h, w), W: (c_o, c_i, kh, kw), b: (c_o,)."""
    if padding:
        X = F.pad(X, (padding,) * 4)               # zeros on all four sides
    c_o, c_i, kh, kw = W.shape
    P = unfold(X, kh, kw)
    Y = W.reshape(c_o, -1) @ P  # (given) (c_o, c_i*kh*kw) kernel matrix times the patch matrix -> (n, c_o, h'*w')
    if b is not None:
        Y = Y + b[:, None]
    return Y.reshape(X.shape[0], c_o, X.shape[2] - kh + 1, X.shape[3] - kw + 1)


P = unfold(torch.arange(9.0).reshape(1, 1, 3, 3), 2, 2)[0]
print("patch matrix (one column per output position):\n", P)
check("unfold on the lecture example (rows = patches)", P.T, [[0, 1, 3, 4], [1, 2, 4, 5], [3, 4, 6, 7], [4, 5, 7, 8]])
check("unfold vs F.unfold", unfold(Xr[None], 2, 3), F.unfold(Xr[None], (2, 3)))
Xr4, Wr, br = torch.randn(2, 3, 8, 7, dtype=torch.float64), torch.randn(5, 3, 3, 3, dtype=torch.float64), torch.randn(5, dtype=torch.float64)
check("conv2d_unfold vs F.conv2d (padding 1)", conv2d_unfold(Xr4, Wr, br, padding=1), F.conv2d(Xr4, Wr, br, padding=1))


In [ ]:
def pool2d(X, size, mode="max"):
    """Pooling with a size x size window and stride 1 on one channel, with loops (d2l 7.5)."""
    Y = torch.zeros((X.shape[0] - size + 1, X.shape[1] - size + 1), dtype=X.dtype)
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            window = X[i:i + size, j:j + size]
            Y[i, j] = ...  ### TODO_STUDENT ### max or mean of the window
    return Y


def pool2d_fast(X, size, mode="max"):
    """Given: non-overlapping size x size windows (stride = size) on (n, c, h, w), with no loops.
    Each window becomes its own pair of axes (3 and 5); then one max/mean over them."""
    n, c, h, w = X.shape
    Xw = X.reshape(n, c, h // size, size, w // size, size)
    return Xw.amax(dim=(3, 5)) if mode == "max" else Xw.mean(dim=(3, 5))


X = torch.arange(9.0).reshape(3, 3)
check("max pooling, lecture example", pool2d(X, 2, "max"), [[4, 5], [7, 8]])
check("average pooling, lecture example", pool2d(X, 2, "avg"), [[2, 3], [5, 6]])
Xr = torch.randn(2, 3, 8, 6, dtype=torch.float64)
check("pool2d vs F.max_pool2d (stride 1)", pool2d(Xr[0, 0], 3), F.max_pool2d(Xr[:1, :1], 3, stride=1)[0, 0])
check("pool2d_fast vs F.max_pool2d", pool2d_fast(Xr, 2), F.max_pool2d(Xr, 2))
check("pool2d_fast vs F.avg_pool2d", pool2d_fast(Xr, 2, "avg"), F.avg_pool2d(Xr, 2))


## (B) LeNet, yours vs PyTorch's [10 points]

LeNet-5 (LeCun et al., 1998; d2l 7.6): two convolution + pooling blocks, then three fully connected
layers. Shapes for a 28×28 image (padding 2 on the first convolution):
1×28×28 → conv 5×5 → 6×28×28 → pool → 6×14×14 → conv 5×5 → 16×10×10 → pool → 16×5×5 → flatten 400 → 120 → 84 → 10.

`lenet(Conv, Pool, Act)` builds it from whichever layer classes you pass in. `MyConv2d` and `MyPool2d`
(given) are thin `nn.Module` wrappers around your `conv2d_unfold` and `pool2d_fast`; they have the same
parameters, with the same names and shapes, as `nn.Conv2d`, so one `load_state_dict` copies a PyTorch
LeNet's weights into yours.


In [ ]:
class MyConv2d(nn.Module):
    """Given: a convolution layer that calls YOUR conv2d_unfold. Same parameters (and init) as nn.Conv2d."""
    def __init__(self, in_channels, out_channels, kernel_size, padding=0):
        super().__init__()
        ref = nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding)
        self.weight, self.bias, self.padding = ref.weight, ref.bias, padding

    def forward(self, X):
        return conv2d_unfold(X, self.weight, self.bias, self.padding)


class MyPool2d(nn.Module):
    """Given: non-overlapping pooling that calls pool2d_fast."""
    def __init__(self, size, mode="avg"):
        super().__init__()
        self.size, self.mode = size, mode

    def forward(self, X):
        return pool2d_fast(X, self.size, self.mode)


def lenet(Conv, Pool, Act):
    """LeNet-5 for 1x28x28 inputs, from the given layer classes: Conv(c_in, c_out, k, padding=...), Pool(2), Act()."""
    layers = [Conv(1, 6, 5, padding=2), Act(), Pool(2)]  # (given) conv 1 -> 6 channels, 5x5, padding 2; activation; 2x2 pooling
    layers += [Conv(6, 16, 5), Act(), Pool(2), nn.Flatten()]  # (given) conv 6 -> 16 channels, 5x5, no padding; activation; 2x2 pooling; flatten
    layers += [nn.Linear(16 * 5 * 5, 120), Act(), nn.Linear(120, 84), Act()]  # (given) fully connected 400 -> 120 -> 84, each followed by the activation
    layers += [nn.Linear(84, 10)]
    return nn.Sequential(*layers)


my_lenet = lambda: lenet(MyConv2d, lambda k: MyPool2d(k, "avg"), nn.Sigmoid)       # 1998: sigmoid + average pooling
torch_lenet = lambda: lenet(nn.Conv2d, nn.AvgPool2d, nn.Sigmoid)

net = torch_lenet()
x = torch.zeros(1, 1, 28, 28)
for layer in net:
    x = layer(x)
    print(f"{type(layer).__name__:10s} output shape {tuple(x.shape)}")
print("LeNet parameters:", n_params(net), "(lecture: 156 + 2,416 + 48,120 + 10,164 + 850 = 61,706)")
assert n_params(net) == 61706

# Checkpoint: copy the PyTorch LeNet's weights into yours; both must give the same outputs (float64, CPU).
mine, ref = my_lenet().double(), torch_lenet().double()
mine.load_state_dict(ref.state_dict())
xb = X_fm[:16].cpu().double()
check("your LeNet vs nn LeNet, same weights", mine(xb), ref(xb), tol=1e-10)


**Train both LeNets, and HW5's MLP for comparison** (given). Same optimizer (Adam, learning rate
$10^{-3}$), batch size 256 and number of epochs for all three. The MLP is HW5 Problem 2's network
(784 → 256 → ReLU → 10), retrained here so that every number in the table comes from the same budget.


In [ ]:
def mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))


P1 = {}
torch.manual_seed(SEED)
m = mlp()
P1["HW5 MLP"] = (m, cached("p1_mlp", m, lambda: train_classifier(m, X_fm, y_fm, X_fm_test, y_fm_test, EPOCHS_FM,
                                                                  1e-3, desc="HW5 MLP")))
torch.manual_seed(SEED)
m = torch_lenet()
P1["LeNet (nn layers)"] = (m, cached("p1_lenet_nn", m, lambda: train_classifier(m, X_fm, y_fm, X_fm_test, y_fm_test,
                                                                                EPOCHS_FM, 1e-3, desc="LeNet nn")))
torch.manual_seed(SEED)
m = my_lenet()
P1["LeNet (your layers)"] = (m, cached("p1_lenet_yours", m, lambda: train_classifier(m, X_fm, y_fm, X_fm_test, y_fm_test,
                                                                                    EPOCHS_FM, 1e-3, desc="LeNet yours")))


def results_table(runs):
    if TAG:
        print(TAG)
    print(f"{'model':30s} {'parameters':>10s} {'epochs':>6s} {'train acc':>9s} {'test acc':>9s} {'s/epoch':>8s}")
    for name, (m, h) in runs.items():
        print(f"{name:30s} {n_params(m):10,d} {len(h['test_acc']):6d} {h['train_acc'][-1]:9.3f} "
              f"{h['test_acc'][-1]:9.3f} {h['seconds'] / len(h['test_acc']):8.1f}")


results_table(P1)
plot_histories({k: h for k, (m, h) in P1.items()}, title="Fashion-MNIST, Adam 1e-3")
plt.show()


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): test accuracy MLP about 0.87–0.89, both LeNets about 0.80–0.83 (the same for yours and `nn`'s); on a GPU your LeNet is several times slower per epoch than `nn`'s.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** Compare the original LeNet with the MLP (accuracy, parameters), and your
LeNet with PyTorch's (accuracy, speed).

*How to think about it.* Look at the table's last epoch and at the curves: is LeNet done learning when
training stops? HW5 Problem 2(F) showed what sigmoids do to gradients. For the speed, think about what
`unfold` does to memory, and how many separate operations each version launches.

*Your answer here.*


## (C) Modernize [10 points]

Two changes that came after 1998: **ReLU** instead of sigmoid (no saturation for positive inputs, so
gradients don't shrink layer by layer) and **max pooling** instead of average pooling (keeps the
strongest response in each window). Use the `nn` layers from here on: they compute the same thing as
yours, faster.

The given cell after the training also tests **robustness to shifts**: every test image is moved right
by a few pixels (filling with background), and the trained MLP and LeNet are re-scored.


In [ ]:
torch.manual_seed(SEED)
modern_lenet = ...  ### TODO_STUDENT ### the same LeNet with ReLU activations and max pooling (nn layers)
m = modern_lenet()
P1["LeNet, ReLU + max pooling"] = (m, cached("p1_lenet_modern", m, lambda: train_classifier(
    m, X_fm, y_fm, X_fm_test, y_fm_test, EPOCHS_FM, 1e-3, desc="LeNet modern")))
results_table(P1)
plot_histories({k: h for k, (m, h) in P1.items()}, title="Fashion-MNIST, Adam 1e-3")
plt.show()


def shift_right(X, d):
    """Move every image d pixels to the right; the uncovered columns become background (0)."""
    if d == 0:
        return X
    Y = torch.zeros_like(X)
    Y[..., d:] = X[..., :-d]
    return Y


SHIFTS = [0, 1, 2, 3, 4]
shift_acc = {name: [accuracy(P1[name][0], shift_right(X_fm_test, d), y_fm_test) for d in SHIFTS]
             for name in ["HW5 MLP", "LeNet, ReLU + max pooling"]}
print(TAG + "test accuracy on images shifted right by d pixels")
print("d =       " + "".join(f"{d:7d}" for d in SHIFTS))
for name, accs in shift_acc.items():
    print(f"{name:10.10s}" + "".join(f"{a:7.3f}" for a in accs))
SUMMARY["P1 test acc"] = {k: round(h["test_acc"][-1], 3) for k, (m, h) in P1.items()}
SUMMARY["P1 shifted 2px"] = {k: round(v[2], 3) for k, v in shift_acc.items()}


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): modernized LeNet about 0.88–0.90, level with or above the MLP; shifted by 2 px: MLP about 0.5–0.6, LeNet about 0.75–0.8.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** How does the modernized LeNet compare with the MLP now, and what happens when
the test images are shifted?

*How to think about it.* Compare test accuracy *and* parameter count. For the shifts: an MLP has one
weight per pixel position; a convolution uses the same kernel at every position. Which of the two
notices that an image moved? Why might LeNet still lose accuracy for larger shifts?

*Your answer here.*


## (D) Batch normalization, from scratch [15 points]

Batch norm (d2l 8.5) standardizes each channel over the minibatch, then lets the network rescale it:
$\hat x = (x-\hat\mu_B)/\sqrt{\hat\sigma_B^2+\epsilon}$, output $\gamma\,\hat x+\beta$, with learned $\gamma$
and $\beta$ per channel. For a convolution's output (n, c, h, w), the mean and variance of each channel
are taken over the batch **and all image positions** (n·h·w values); for a fully connected layer's
output (n, c), over the batch.

* **Training mode** uses the batch's own mean and (biased) variance, and updates **running averages**:
  `running = (1 - momentum) * running + momentum * batch_value` with PyTorch's `momentum = 0.1`.
* **Evaluation mode** uses the running averages, so a prediction doesn't depend on which other images
  are in the batch.

PyTorch convention to match: `running_var` stores the **unbiased** variance (divide by n − 1); the given
line does that.


In [ ]:
def batch_norm(X, gamma, beta, running_mean, running_var, training, eps=1e-5, momentum=0.1):
    """X: (n, c, h, w) or (n, c). Updates running_mean / running_var in place in training mode."""
    dims = (0, 2, 3) if X.dim() == 4 else (0,)       # per channel: over the batch (and the image positions)
    shape = (1, -1, 1, 1) if X.dim() == 4 else (1, -1)
    if training:
        mean = ...  ### TODO_STUDENT ### the batch mean of each channel
        var = ...  ### TODO_STUDENT ### the batch variance of each channel (biased: divide by the number of values)
        n = X.numel() / X.shape[1]
        with torch.no_grad():
            ...  ### TODO_STUDENT ### update the running mean in place
            running_var.mul_(1 - momentum).add_(momentum * var * n / (n - 1))   # PyTorch stores the unbiased variance
    else:
        mean, var = ...  ### TODO_STUDENT ### evaluation mode: the running averages
    X_hat = ...  ### TODO_STUDENT ### standardize
    return ...  ### TODO_STUDENT ### scale by gamma, shift by beta


class MyBatchNorm(nn.Module):
    """Given: a batch-norm layer around YOUR batch_norm, with nn.BatchNorm's parameter and buffer names."""
    def __init__(self, num_features):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(num_features))     # gamma
        self.bias = nn.Parameter(torch.zeros(num_features))      # beta
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))
        self.register_buffer("num_batches_tracked", torch.tensor(0))   # unused; kept so state_dicts match

    def forward(self, X):
        return batch_norm(X, self.weight, self.bias, self.running_mean, self.running_var, self.training)


# Checkpoint 1: the lecture example (3 examples x 5 features, nn.BatchNorm1d-style).
x = torch.tensor([[1.0, 2, 3, 4, 5], [10, 20, 30, 40, 50], [7, 14, 21, 28, 35]], dtype=torch.float64)
bn = MyBatchNorm(5).double()
out = bn(x)
print(out)
check("BN on the lecture example (every column -> -1.336, 1.069, 0.267)", out[:, 0], [-1.3363, 1.0690, 0.2673], tol=1e-4)
check("running_mean after one step", bn.running_mean, [0.6, 1.2, 1.8, 2.4, 3.0], tol=1e-12)
check("running_var[0] after one step (unbiased: 0.9 + 0.1 * 21)", bn.running_var[0], 3.0, tol=1e-12)

# Checkpoint 2: against nn.BatchNorm2d / nn.BatchNorm1d, training mode (output + buffers) and evaluation mode.
for X, ref in [(torch.randn(8, 3, 5, 4, dtype=torch.float64) * 3 + 1, nn.BatchNorm2d(3).double()),
               (torch.randn(8, 6, dtype=torch.float64) * 2 - 1, nn.BatchNorm1d(6).double())]:
    mine = MyBatchNorm(X.shape[1]).double()
    with torch.no_grad():
        ref.weight.uniform_(0.5, 1.5); ref.bias.uniform_(-1, 1)
    mine.load_state_dict(ref.state_dict())
    for _ in range(3):                                  # three training steps
        check(f"{type(ref).__name__} training mode", mine(X), ref(X), tol=1e-10)
    check(f"{type(ref).__name__} running_mean", mine.running_mean, ref.running_mean, tol=1e-12)
    check(f"{type(ref).__name__} running_var", mine.running_var, ref.running_var, tol=1e-12)
    mine.eval(); ref.eval()
    check(f"{type(ref).__name__} evaluation mode", mine(X[:2]), ref(X[:2]), tol=1e-10)


**Batch norm in LeNet.** `bn_lenet(Norm)` (given) is the modernized LeNet with a batch-norm layer
after every convolution and every hidden fully connected layer, before the activation (d2l 8.5.4). It
uses **your** `MyBatchNorm`. Four runs: LeNet without and with batch norm, at the learning rate used so
far ($10^{-3}$) and at a 30× higher one ($3\cdot10^{-2}$).


In [ ]:
def bn_lenet(Norm):
    return nn.Sequential(
        nn.Conv2d(1, 6, 5, padding=2), Norm(6), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(6, 16, 5), Norm(16), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(),
        nn.Linear(400, 120), Norm(120), nn.ReLU(),
        nn.Linear(120, 84), Norm(84), nn.ReLU(),
        nn.Linear(84, 10))


P1_BN = {}
for lr in [1e-3, 3e-2]:
    for name, make in [("LeNet", modern_lenet), ("BN-LeNet", lambda: bn_lenet(MyBatchNorm))]:
        torch.manual_seed(SEED)
        m = make()
        P1_BN[f"{name}, lr {lr:g}"] = (m, cached(f"p1_{name}_lr{lr:g}", m, lambda: train_classifier(
            m, X_fm, y_fm, X_fm_test, y_fm_test, EPOCHS_FM, lr, desc=f"{name} lr {lr:g}")))
results_table(P1_BN)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, lr in zip(axes, [1e-3, 3e-2]):
    plot_histories({k: h for k, (m, h) in P1_BN.items() if k.endswith(f"{lr:g}")}, ax=ax, title=f"learning rate {lr:g}")
plt.tight_layout(); plt.show()
print(TAG + "test accuracy after the FIRST epoch:", {k: round(h["test_acc"][0], 3) for k, (m, h) in P1_BN.items()})
SUMMARY["P1 BN"] = {k: (round(h["test_acc"][0], 3), round(h["test_acc"][-1], 3)) for k, (m, h) in P1_BN.items()}


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): first-epoch test accuracy LeNet about 0.75–0.8 vs BN-LeNet about 0.86–0.88; at lr 3e-2 plain LeNet ends lower (about 0.84–0.87), BN-LeNet about 0.90–0.91.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** Compare the four runs: speed of learning (look at the first epochs) and the
effect of the higher learning rate.

*How to think about it.* Read the first-epoch accuracies printed under the plots. Then ask what a large
step in an early layer does to the scale of the inputs every later layer sees — with and without a layer
that re-standardizes them at every step.

*Your answer here.*


---
# PROBLEM 2 — Deeper CNNs on Natural Images (CIFAR-10) [50 points]

CIFAR-10: 32×32 colour photos in 10 classes (airplane, car, bird, cat, deer, dog, frog, horse, ship,
truck), 50,000 for training and 10,000 for testing. Much harder than Fashion-MNIST: objects vary in pose,
size, position, colour and background.

*Library calls:* `torchvision.models.resnet18`, `torchvision.transforms.v2`.

The images are standardized per colour channel with the training set's mean and standard deviation and
kept on the GPU. All runs use AdamW with a *one-cycle* learning-rate schedule (warm up, then decay to
about 0; standard for short CNN training) and batch size 128.


In [ ]:
def load_cifar(train):
    if not os.path.isdir(f"{TORCH_DATA}/cifar-10-batches-py"):
        course_file("torchvision/cifar-10-python.tar.gz",       # the original server is slow
                    "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz")
    ds = torchvision.datasets.CIFAR10(root=TORCH_DATA, train=train, download=True)
    return torch.tensor(ds.data).permute(0, 3, 1, 2).contiguous().float() / 255, torch.tensor(ds.targets)


X_c, y_c = load_cifar(train=True)
X_c_test, y_c_test = load_cifar(train=False)
CIFAR_MEAN = X_c.mean(dim=(0, 2, 3), keepdim=True)
CIFAR_STD = X_c.std(dim=(0, 2, 3), keepdim=True)
X_c, X_c_test = ((X_c - CIFAR_MEAN) / CIFAR_STD).to(TRAIN_DEVICE), ((X_c_test - CIFAR_MEAN) / CIFAR_STD).to(TRAIN_DEVICE)
y_c, y_c_test = y_c.to(TRAIN_DEVICE), y_c_test.to(TRAIN_DEVICE)
CIFAR_CLASSES = ["airplane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
X_c, y_c = X_c[:C["cifar_train"]], y_c[:C["cifar_train"]]                    # dev: subsets
X_c_test, y_c_test = X_c_test[:C["cifar_test"]], y_c_test[:C["cifar_test"]]
EPOCHS_C = C["cifar_epochs"]
print("CIFAR-10:", tuple(X_c.shape), tuple(X_c_test.shape))


def show_images(X, titles=None, n=8, title=None):
    imgs = (X[:n].cpu() * CIFAR_STD + CIFAR_MEAN).clamp(0, 1).permute(0, 2, 3, 1)
    fig, axes = plt.subplots(1, n, figsize=(1.4 * n, 1.7))
    for i, ax in enumerate(axes):
        ax.imshow(imgs[i]); ax.axis("off")
        if titles is not None:
            ax.set_title(titles[i], fontsize=8)
    if title:
        fig.suptitle(title, fontsize=9)
    plt.show()


show_images(X_c, [CIFAR_CLASSES[i] for i in y_c[:8].tolist()])


## (A) Residual block [20 points]

A residual block (He et al., 2016; d2l 8.6) computes $f(x) = \mathrm{ReLU}(x + g(x))$, where $g$ is two
3×3 convolutions with batch norm. The **skip connection** $x + \cdot$ gives the signal (and, going
backwards, the gradient) a direct path that skips the convolutions: whatever $g$ learns, $x$ itself
still reaches the next block, so a block can refine its input without having to re-learn it. When the block changes the
number of channels or the image size (stride 2), $x$ no longer has the shape of $g(x)$, and a **1×1
convolution** (with the same stride) reshapes it.

Layout of $g$: conv 3×3 (stride `stride`, padding 1) → BN → ReLU → conv 3×3 (padding 1) → BN. The
convolutions have no bias (`bias=False`): the batch norm right after them has its own shift β.


In [ ]:
class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)  # (given) 3x3 conv, in -> out channels, stride `stride`, padding 1, no bias
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)  # (given) 3x3 conv, out -> out channels, padding 1, no bias
        self.bn2 = nn.BatchNorm2d(out_channels)
        reshape = in_channels != out_channels or stride != 1
        self.conv3 = nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False) if reshape else None  # (given) the 1x1 conv (same stride) when the shapes change, else None

    def forward(self, X):
        Y = ...  ### TODO_STUDENT ### first conv -> BN -> ReLU
        Y = ...  ### TODO_STUDENT ### second conv -> BN (no ReLU yet)
        shortcut = ...  ### TODO_STUDENT ### X itself, or X through the 1x1 conv
        return ...  ### TODO_STUDENT ### the skip connection, then ReLU


x = torch.randn(4, 16, 32, 32)
for blk, shape in [(Residual(16, 16), (4, 16, 32, 32)), (Residual(16, 32, stride=2), (4, 32, 16, 16))]:
    y = blk(x)
    assert tuple(y.shape) == shape, f"output shape {tuple(y.shape)}, expected {shape}"
    print(f"CHECK passed: Residual{(blk.conv1.in_channels, blk.conv1.out_channels, blk.conv1.stride[0])} -> {tuple(y.shape)}")
# If the last BN's scale gamma is 0, g(x) = 0 and the block is just ReLU(shortcut): the skip path alone.
blk = Residual(16, 16).double().eval()
with torch.no_grad():
    blk.bn2.weight.zero_(); blk.bn2.bias.zero_()
check("Residual with g(x) = 0 is ReLU(x)", blk(x.double()), F.relu(x.double()), tol=1e-12)


# With a NONZERO branch: fixed weights (no randomness), compared with the solution's output.
def fixed_block(c_in, c_out, stride):
    blk = Residual(c_in, c_out, stride).double().eval()   # eval: BN uses running mean 0, variance 1
    with torch.no_grad():
        for k, (name, p) in enumerate(sorted(blk.named_parameters())):
            p.copy_(0.5 * torch.sin(torch.arange(p.numel(), dtype=torch.float64) + k).reshape(p.shape))
    return blk


x = torch.cos(torch.arange(2 * 3 * 4 * 4, dtype=torch.float64)).reshape(2, 3, 4, 4)
for (c_in, c_out, stride), expected in [((3, 3, 1), (39.145351, 39.523624)), ((3, 4, 2), (5.254389, 2.459709))]:
    y = fixed_block(c_in, c_out, stride)(x)
    check(f"Residual({c_in}, {c_out}, stride={stride}) with fixed weights: sum and sum of squares of the output",
          [y.sum().item(), (y ** 2).sum().item()], expected, tol=1e-5)
print("(if this fails: order conv1 -> bn1 -> ReLU -> conv2 -> bn2, then + shortcut, then ReLU; bias=False)")


**A small ResNet vs the batch-normalized LeNet** (both given). The ResNet: a 3×3 stem convolution
with 32 channels, then five residual blocks (32 → 32, 32 → 64 with stride 2, 64 → 64, 64 → 128 with
stride 2, 128 → 128), global average pooling and one linear layer: 11 convolutions deep. The BN-LeNet is
Problem 1(D)'s network adapted to 3 input channels and 32×32 images (with `nn.BatchNorm`).


In [ ]:
def cifar_bn_lenet():
    return nn.Sequential(
        nn.Conv2d(3, 6, 5, padding=2), nn.BatchNorm2d(6), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(6, 16, 5), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(),
        nn.Linear(16 * 6 * 6, 120), nn.BatchNorm1d(120), nn.ReLU(),
        nn.Linear(120, 84), nn.BatchNorm1d(84), nn.ReLU(),
        nn.Linear(84, 10))


def small_resnet(widths=C["resnet_widths"]):          # full: (32, 64, 128); dev: half as wide
    a, b, c = widths
    return nn.Sequential(
        nn.Conv2d(3, a, 3, padding=1, bias=False), nn.BatchNorm2d(a), nn.ReLU(),
        Residual(a, a), Residual(a, b, stride=2), Residual(b, b), Residual(b, c, stride=2), Residual(c, c),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 10))


P2 = {}
for name, make in [("BN-LeNet", cifar_bn_lenet), ("small ResNet", small_resnet)]:
    torch.manual_seed(SEED)
    m = make()
    P2[name] = (m, cached(f"p2_{name}", m, lambda: train_classifier(m, X_c, y_c, X_c_test, y_c_test, EPOCHS_C, 2e-3,
                                                                    batch_size=128, onecycle=True, desc=name)))
results_table(P2)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
plot_histories({k: h for k, (m, h) in P2.items()}, ax=axes[0], title="test accuracy")
plot_histories({k: h for k, (m, h) in P2.items()}, key="train_acc", ax=axes[1], title="training accuracy",
               ylabel="training accuracy")
plt.tight_layout(); plt.show()


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): test accuracy BN-LeNet about 0.66–0.70, small ResNet about 0.84–0.87 (training accuracy above 0.98).
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** Compare the two networks (accuracy, parameters, time).

*How to think about it.* Look at both plots: does each network fit its *training* data? What do the
extra layers buy on natural images, and what do the skip connections do for training a deeper network
(think of HW5 Problem 2(F))? Note the train/test gap for (B).

*Your answer here.*


## (B) Data augmentation [15 points]

A photo of a cat is still a cat when it is mirrored left–right or moved by a few pixels. **Augmentation**
uses this: every time a training image is used, it is randomly flipped (probability 1/2) and randomly
cropped (pad 4 pixels of zeros on every side, then cut a random 32×32 window). The network rarely sees
exactly the same image twice, which makes memorizing the training set much harder. Only the **training** batches are augmented;
test images are used as they are.

Write `augment` for a batch on the GPU: one random decision per image. (In the pictures below the
padding shows as a grey border: after standardization, 0 is the average colour.)


In [ ]:
def augment(X, pad=4):
    """Random horizontal flip and random crop (zero padding `pad`) of each image in a batch (n, c, h, w)."""
    n, c, h, w = X.shape
    flip = torch.rand(n, device=X.device) < 0.5
    X = torch.where(flip[:, None, None, None], X.flip(dims=[3]), X)  # (given) the flipped image where flip is True, the original elsewhere
    Xp = F.pad(X, (pad, pad, pad, pad))  # (given) zero padding on all four sides
    top = torch.randint(0, 2 * pad + 1, (n,)).tolist()
    left = torch.randint(0, 2 * pad + 1, (n,)).tolist()
    return torch.stack([Xp[k, :, top[k]:top[k] + h, left[k]:left[k] + w] for k in range(n)])  # (given) for each image k, its h x w window starting at (top[k], left[k])


xb = X_c[:64]
out = augment(xb)
assert out.shape == xb.shape, f"augment changed the shape: {tuple(out.shape)}"
flipped_or_not = [(torch.equal(o, x) or torch.equal(o, x.flip(dims=[2]))) for o, x in zip(augment(xb, pad=0), xb)]
assert all(flipped_or_not), "with pad=0, every output must be the image itself or its mirror image"
print(f"CHECK passed: with pad=0 every image is itself or its mirror ({sum(not torch.equal(o, x) for o, x in zip(augment(xb, pad=0), xb))} of 64 flipped in one draw)")


def find_window(img, out, pad):
    """(flipped, top, left) such that out is that 32x32 window of img (or of its mirror), zero-padded; else None."""
    c, h, w = img.shape
    for flipped, src in [(False, img), (True, img.flip(dims=[2]))]:
        big = torch.zeros(c, h + 2 * pad, w + 2 * pad)
        big[:, pad:pad + h, pad:pad + w] = src
        for top in range(2 * pad + 1):
            for left in range(2 * pad + 1):
                if torch.equal(big[:, top:top + h, left:left + w], out):
                    return flipped, top, left
    return None


xc = xb[:16].cpu()
torch.manual_seed(0)
found = [find_window(x, o, 4) for x, o in zip(xc, augment(xc))]
assert all(f is not None for f in found), "some outputs are not a 32x32 window of the padded (flipped) image"
torch.manual_seed(0)                         # the same random draws as inside augment
drawn_flip = (torch.rand(16) < 0.5).tolist()
drawn_top, drawn_left = torch.randint(0, 9, (16,)).tolist(), torch.randint(0, 9, (16,)).tolist()
assert [f[0] for f in found] == drawn_flip, "the flipped images are not the ones where flip is True"
assert [(f[1], f[2]) for f in found] == list(zip(drawn_top, drawn_left)), "the crop windows don't start at (top[k], left[k])"
print(f"CHECK passed: every output is the drawn window of the padded image ({sum(drawn_flip)} of 16 flipped, "
      f"{sum((t, l) != (4, 4) for t, l in zip(drawn_top, drawn_left))} of 16 moved)")
torch.manual_seed(1)
show_images(xb, title="original training images")
show_images(augment(xb), title="the same images through your augment")
show_images(torch.stack([augment(xb[:1])[0] for _ in range(8)]), title="your augment: 8 random draws of the first image")
from torchvision.transforms import v2
library_aug = v2.Compose([v2.RandomCrop(32, padding=4), v2.RandomHorizontalFlip()])
show_images(torch.stack([library_aug(xb[0].cpu()) for _ in range(8)]),
            title="torchvision.transforms.v2 (RandomCrop + RandomHorizontalFlip): 8 draws, the same kind of variety")


In [ ]:
torch.manual_seed(SEED)
m = small_resnet()
P2["small ResNet + augmentation"] = (m, cached("p2_resnet_aug", m, lambda: train_classifier(
    m, X_c, y_c, X_c_test, y_c_test, EPOCHS_C, 2e-3, batch_size=128, onecycle=True, augment=augment, desc="ResNet + aug")))
results_table(P2)
print("\n" + TAG + "train/test gap (training accuracy - test accuracy):")
for name in ["small ResNet", "small ResNet + augmentation"]:
    h = P2[name][1]
    print(f"  {name:30s} {h['train_acc'][-1]:.3f} - {h['test_acc'][-1]:.3f} = {h['train_acc'][-1] - h['test_acc'][-1]:.3f}")
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, style in [("small ResNet", "-"), ("small ResNet + augmentation", "--")]:
    h = P2[name][1]
    ep = range(1, len(h["test_acc"]) + 1)
    ax.plot(ep, h["train_acc"], "C0" + style, label=f"{name}: train")
    ax.plot(ep, h["test_acc"], "C1" + style, label=f"{name}: test")
ax.set_xlabel("epoch"); ax.set_ylabel("accuracy"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.show()
SUMMARY["P2"] = {k: (round(h["train_acc"][-1], 3), round(h["test_acc"][-1], 3)) for k, (m, h) in P2.items()}


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): train − test gap about 0.12–0.15 without augmentation, about 0.03–0.06 with it; test accuracy up by about 0.01–0.03.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** How did augmentation change the training accuracy, the test accuracy and the
gap?

*How to think about it.* Use the printed gap lines. Why can augmentation reduce overfitting: what does
the network have to fit now, instead of 50,000 fixed images? What knowledge about images does the
augmentation add that the network doesn't have?

*Your answer here.*


## (C) Transfer learning [15 points]

Labelled data is often scarce. Here: only **5,000** training images (500 per class). Three ways to use
them:

1. **From scratch:** the small ResNet of (A), with augmentation (30 epochs: the subset is 10× smaller).
2. **Linear probe:** ResNet-18 *pretrained on ImageNet* (1.3 million photos, 1000 classes;
   `torchvision.models.resnet18`). Freeze all its weights and train only a new last layer (512 → 10). The
   network is used as a fixed *feature extractor*; the given code computes the 512 features of every
   image once and then trains the linear layer on them.
3. **Fine-tuning:** the same pretrained ResNet-18 with a new last layer, but now **all** weights train
   (small learning rate, 5 epochs, augmentation).

ResNet-18 was trained on 224×224 images; our 32×32 images are **resized to 128×128** (bilinear) and
normalized with ImageNet's channel statistics. (At 32×32 the pretrained features are poor; 128 gives
the same accuracy as 224 in a third of the time.)


In [ ]:
RESIZE = C["resize"]              # full: 128; dev: 64
SUB_EPOCHS, PROBE_EPOCHS, FT_EPOCHS = C["sub_epochs"], C["probe_epochs"], C["ft_epochs"]
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=TRAIN_DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=TRAIN_DEVICE).view(1, 3, 1, 1)
_cm, _cs = CIFAR_MEAN.to(TRAIN_DEVICE), CIFAR_STD.to(TRAIN_DEVICE)


def to_imagenet(X):
    """CIFAR-standardized 32x32 batch -> RESIZE x RESIZE, standardized the way ResNet-18 was trained."""
    X = F.interpolate(X * _cs + _cm, size=RESIZE, mode="bilinear", align_corners=False)
    return (X - IMAGENET_MEAN) / IMAGENET_STD


g = torch.Generator().manual_seed(SEED)
sub = torch.cat([torch.nonzero(y_c.cpu() == k).flatten()[torch.randperm(int((y_c == k).sum()), generator=g)[:C["sub_per_class"]]]
                 for k in range(10)]).to(TRAIN_DEVICE)
X_sub, y_sub = X_c[sub], y_c[sub]
print("training subset:", tuple(X_sub.shape), "class counts:", torch.bincount(y_sub).tolist())

P2C = {}
torch.manual_seed(SEED)
m = small_resnet()
P2C["small ResNet from scratch"] = (m, cached("p2c_scratch", m, lambda: train_classifier(
    m, X_sub, y_sub, X_c_test, y_c_test, SUB_EPOCHS, 2e-3, batch_size=128, onecycle=True, augment=augment,
    desc="scratch, subset")))


In [ ]:
# 2. Linear probe: frozen pretrained ResNet-18; only a new last layer is trained.
probe_net = torchvision.models.resnet18(weights="IMAGENET1K_V1")
...  ### TODO_STUDENT ### freeze every pretrained weight (no gradients)
probe_net.fc = nn.Identity()          # given: drop ImageNet's 1000-class layer; the network now outputs 512 features
probe_net.to(TRAIN_DEVICE).eval()     # evaluation mode: batch norm uses ImageNet's running statistics


@torch.no_grad()
def features(net, X, batch_size=250):
    return torch.cat([net(to_imagenet(X[i:i + batch_size])) for i in range(0, len(X), batch_size)])


t0 = time.time()
F_sub, F_test = features(probe_net, X_sub), features(probe_net, X_c_test)
feature_seconds = time.time() - t0
print(f"512 features for {len(F_sub) + len(F_test):,} images in {feature_seconds:.0f} s")
torch.manual_seed(SEED)
probe = ...  ### TODO_STUDENT ### the new, trainable last layer: 512 features -> 10 classes
h = dict(cached("p2c_probe", probe, lambda: train_classifier(probe, F_sub, y_sub, F_test, y_c_test, PROBE_EPOCHS, 1e-3,
                                                             batch_size=128, weight_decay=1e-2, desc="linear probe")))
h["seconds"] += feature_seconds
P2C["pretrained ResNet-18, linear probe"] = (probe, h)

# 3. Fine-tuning: the same pretrained network, new last layer, ALL weights train.
torch.manual_seed(SEED)
ft = torchvision.models.resnet18(weights="IMAGENET1K_V1")
ft.fc = nn.Linear(512, 10)            # given: a new last layer for 10 classes (everything stays trainable)
P2C["pretrained ResNet-18, fine-tuned"] = (ft, cached("p2c_finetune", ft, lambda: train_classifier(
    ft, X_sub, y_sub, X_c_test, y_c_test, FT_EPOCHS, 5e-4, batch_size=128, onecycle=True, weight_decay=0.05,
    augment=augment, transform=to_imagenet, desc="fine-tune")))
print(f"\n{TAG}{len(X_sub):,} training images ({RESIZE}x{RESIZE} for ResNet-18); test accuracy on {len(X_c_test):,} test images")
print(f"{'model':40s} {'trained params':>14s} {'epochs':>6s} {'test acc':>9s} {'seconds':>8s}")
for name, (m, h) in P2C.items():
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"{name:40s} {trainable:14,d} {len(h['test_acc']):6d} {h['test_acc'][-1]:9.3f} {h['seconds']:8.0f}")
SUMMARY["P2C"] = {k: round(h["test_acc"][-1], 3) for k, (m, h) in P2C.items()}


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): from scratch about 0.70–0.75, linear probe about 0.79–0.83, fine-tuned about 0.86–0.89.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** Rank the three approaches and explain the ranking.

*How to think about it.* What did ResNet-18 learn from 1.3 million ImageNet photos that a network trained
on 5,000 images must learn from scratch? Then compare the probe and fine-tuning: the probe keeps
ImageNet's features exactly as they are — are blurry, upsampled 32×32 thumbnails like ImageNet photos?

*Your answer here.*


---
# PROBLEM 3 — A Recurrent Language Model (TinyStories) [60 points]

A **language model** predicts the next word from the words so far. It's the simplest task that forces a
network to remember: to predict the word after "Lily gave the ball to", the model has to know who has the
ball. And it is how large language models are trained.

*Library calls:* `torch.nn.RNN`, `torch.nn.GRU`, `torch.nn.LSTM`, `torch.nn.utils.clip_grad_norm_`.

**Data.** TinyStories (Eldan & Li, 2023): short children's stories written (by GPT-3.5/4) with the
vocabulary of a 3-year-old, so a small model trained for a few minutes can write fluent text. We use the
official 19 MB validation file (about 22,000 stories): the last 5% of the stories are our validation
set. It is downloaded into `../../data/tinystories/` on first run.


## From text to training examples (read this first; nothing to do)

A network computes with numbers, so text goes through four steps:

1. **Tokens.** Split the text into *tokens*: here lower-case words and punctuation marks
   (`"Lily's dog ran."` → `lily's`, `dog`, `ran`, `.`). Every story ends with an `<eos>` token.
2. **IDs.** The **vocabulary** lists every word seen at least 10 times in the training stories and gives
   it an integer ID; any rarer word becomes `<unk>` ("unknown"). TinyStories' vocabulary is small:
   about 5,300 words cover 99.5% of the text.
3. **Embedding lookup.** The first layer, `nn.Embedding(V, E)`, is a table with one learned row of E
   numbers per word: word ID 17 → row 17. That vector is what the network reads. (It is the same as
   multiplying a one-hot vector by a V × E weight matrix, without building the one-hot vector.)
4. **Next-token targets.** Cut the token stream into windows of $T + 1$ consecutive IDs. The input is the
   first $T$, the target the last $T$: **the target at each position is the next token**. One window
   gives $T$ training examples at once, each with a different amount of context.

The model outputs, at every position, a score (*logit*) for each of the V words; softmax turns the scores
into probabilities and the loss is the cross-entropy with the true next word, averaged over all
positions — exactly HW5's classifier, with V classes. **Perplexity** = exp(average cross-entropy) is the
number the field reports: a model with perplexity 20 is as uncertain, on average, as if it had to choose
uniformly among 20 words. Guessing uniformly among all V words gives perplexity V.


In [ ]:
lm = lm_common.TinyStories(f"{DATA}/tinystories", batch_size=C["lm_batch"], num_steps=C["lm_len"],
                           min_freq=C["lm_min_freq"])           # dev: a smaller vocabulary (rarer words -> <unk>)
print(lm.describe())
text = "Lily's dog ran to the park. He was very happy!"
tokens = lm_common.tokenize(text)
ids = lm.vocab[tokens]
print("\ntext:  ", text)
print("tokens:", tokens)
print("IDs:   ", ids)
emb = nn.Embedding(len(lm.vocab), 4)
print("embedding of the first 3 tokens (a learned table; random before training):\n", emb(torch.tensor(ids[:3])).detach())
X, Y = next(iter(lm.get_dataloader(train=True)))
print("\none training batch: X", tuple(X.shape), " Y", tuple(Y.shape), "(batch, steps)")
print("X[0, :12] =", lm_common.detokenize(lm.vocab.to_tokens(X[0, :12].tolist())))
print("Y[0, :12] =", lm_common.detokenize(lm.vocab.to_tokens(Y[0, :12].tolist())), "  <- the same, shifted by one")
V = len(lm.vocab)
print(f"\nuniform guessing: perplexity {V} (cross-entropy ln {V} = {math.log(V):.3f} nats)")


## (A) An RNN from scratch, step by step [30 points]

The recurrent cell (d2l 9.4), in the row-vector notation of class, for a batch $X_t$ (n × E) at time t:

$$H_t = \tanh\!\left(X_t W_{xh} + H_{t-1} W_{hh} + b_h\right),$$

with $W_{xh}$ (E × H), $W_{hh}$ (H × H), $b_h$ (H). **The same weights are used at every time step**
(weight sharing over time, as a convolution shares them over positions). $H_t$ summarizes everything read
so far; an output layer turns it into next-word scores. Sequences are **time-major**, as in the book and
`nn.RNN`: shape (T, n, E).

Four checkpointed steps: one cell step (against the lecture's hand-computed numbers) → the whole sequence
→ the same outputs as `nn.RNN` (the given helper copies your weights in) → gradient clipping.


In [ ]:
def rnn_step(X_t, H_prev, W_xh, W_hh, b_h):
    """One step of the RNN cell: X_t (n, E), H_prev (n, H) -> H_t (n, H)."""
    return ...  ### TODO_STUDENT ### H_t = tanh(X_t W_xh + H_{t-1} W_hh + b_h)


# Checkpoint: the lecture's hand-computed step. net = (0.35, -0.05), H_t = tanh(net).
H_t = rnn_step(torch.tensor([[1.0, 0.0]]), torch.tensor([[0.5, -0.5]]),
               torch.tensor([[0.1, 0.2], [0.3, 0.4]]), 0.5 * torch.eye(2), torch.zeros(2))
print("H_t =", H_t)
check("one RNN step, lecture example", H_t, [[0.3364, -0.0500]], tol=1e-4)


class MyRNN(nn.Module):
    """A one-layer tanh RNN with YOUR cell. Same interface as nn.RNN (time-major):
    forward(X (T, n, E), H0 (1, n, H) or None) -> outputs (T, n, H), H_T (1, n, H)."""

    def __init__(self, input_size, hidden_size):
        super().__init__()
        bound = 1 / math.sqrt(hidden_size)          # nn.RNN's initialization
        self.W_xh = nn.Parameter(torch.empty(input_size, hidden_size).uniform_(-bound, bound))
        self.W_hh = nn.Parameter(torch.empty(hidden_size, hidden_size).uniform_(-bound, bound))
        self.b_h = nn.Parameter(torch.empty(hidden_size).uniform_(-bound, bound))
        self.hidden_size = hidden_size

    def forward(self, X, H0=None):
        H = X.new_zeros(X.shape[1], self.hidden_size) if H0 is None else H0[0]
        outputs = []
        for X_t in X:                              # loop over time
            H = ...  ### TODO_STUDENT ### one cell step: the new state from X_t and the previous state
            outputs.append(H)
        return torch.stack(outputs), H.unsqueeze(0)


def copy_rnn_to_torch(mine, ref):
    """Given: nn.RNN computes tanh(x W_ih^T + b_ih + h W_hh^T + b_hh), so W_ih = W_xh^T, W_hh^T = W_hh, b_ih + b_hh = b_h."""
    with torch.no_grad():
        ref.weight_ih_l0.copy_(mine.W_xh.T); ref.weight_hh_l0.copy_(mine.W_hh.T)
        ref.bias_ih_l0.copy_(mine.b_h); ref.bias_hh_l0.zero_()


mine, ref = MyRNN(5, 7).double(), nn.RNN(5, 7).double()
copy_rnn_to_torch(mine, ref)
Xs, H0 = torch.randn(11, 3, 5, dtype=torch.float64), torch.randn(1, 3, 7, dtype=torch.float64)
(out_m, h_m), (out_r, h_r) = mine(Xs, H0), ref(Xs, H0)
check("MyRNN outputs vs nn.RNN (11 steps)", out_m, out_r, tol=1e-12)
check("MyRNN final state vs nn.RNN", h_m, h_r, tol=1e-12)


**Gradient clipping.** Backpropagation through time multiplies by $W_{hh}$ (and the tanh slopes)
once per step back, so over many steps gradients can *explode*. Clipping rescales the **whole** gradient
(all parameters together) when its norm is above a threshold θ:
$g \leftarrow \min\!\left(1, \theta/\lVert g\rVert\right) g$. The direction stays the same; only the step
length is capped.


In [ ]:
def clip_gradients(params, max_norm):
    """Rescale all gradients together so that their global norm is at most max_norm; returns the norm before."""
    grads = [p.grad for p in params if p.grad is not None]
    norm = ...  ### TODO_STUDENT ### the global norm: sqrt of the sum of squares of ALL gradient entries
    scale = ...  ### TODO_STUDENT ### min(1, max_norm / norm)  (the 1e-6 avoids dividing by 0, as PyTorch does)
    for g in grads:
        g.mul_(scale)
    return norm


# Checkpoint 1: the lecture's numbers, g = (3, 4) has norm 5.
p = nn.Parameter(torch.zeros(2)); p.grad = torch.tensor([3.0, 4.0])
clip_gradients([p], 1.0)
check("clip (3, 4) to norm 1", p.grad, [0.6, 0.8], tol=1e-6)
p.grad = torch.tensor([3.0, 4.0]); clip_gradients([p], 10.0)
check("norm 5 < 10: unchanged", p.grad, [3.0, 4.0], tol=1e-12)
# Checkpoint 2: against clip_grad_norm_ on a real model's gradients.
net_a = nn.Sequential(nn.Linear(10, 20), nn.Tanh(), nn.Linear(20, 3)).double()
net_b = nn.Sequential(nn.Linear(10, 20), nn.Tanh(), nn.Linear(20, 3)).double()
net_b.load_state_dict(net_a.state_dict())
xb = torch.randn(32, 10, dtype=torch.float64)
for net in (net_a, net_b):
    (net(xb) ** 2).sum().mul(100).backward()
clip_gradients(list(net_a.parameters()), 1.0)
nn.utils.clip_grad_norm_(net_b.parameters(), 1.0)
check("clip_gradients vs clip_grad_norm_", torch.cat([p.grad.flatten() for p in net_a.parameters()]),
      torch.cat([p.grad.flatten() for p in net_b.parameters()]), tol=1e-12)


**The baseline that ignores context.** A *unigram* model predicts every word with its frequency in
the training text, whatever came before: $p(w) = \text{count}(w) / \text{number of training tokens}$. Its
perplexity on the validation text is how well you can do *without any memory*. Every model below should
beat it by far.


In [ ]:
counts = ...  ### TODO_STUDENT ### how often each word ID occurs in the training tokens
p_unigram = ...  ### TODO_STUDENT ### unigram probabilities
unigram_ppl = ...  ### TODO_STUDENT ### exp(average negative log-probability of the validation tokens)
print(f"{TAG}unigram perplexity: {unigram_ppl:.1f}   (uniform over {V} words: {V})")
if MODE == "full":
    assert 250 < unigram_ppl < 350, "expected about 303"
print("most frequent tokens:", lm.vocab.to_tokens(torch.argsort(counts, descending=True)[:12].tolist()))
SUMMARY["P3 unigram ppl"] = round(unigram_ppl, 1)


**Train your RNN against `nn.RNN`** (given). `RNNLM` is embedding → recurrent layer → linear output
layer, with embedding size 256 and 512 hidden units (about 4.4 million parameters, most of them in the
embedding and output tables). Same data stream, optimizer (Adam, learning rate $2\cdot10^{-3}$, clipping
at 1) and number of steps for both; your RNN uses your `clip_gradients`. The two are initialized
independently (and `nn.RNN` has a second bias vector), so expect similar, not identical, numbers.


In [ ]:
EMBED, HIDDEN = C["lm_embed"], C["lm_hidden"]        # full: 256, 512
LM_STEPS = C["lm_steps"]


class RNNLM(nn.Module):
    """Embedding -> recurrent layer (time-major) -> linear layer. X (n, T) -> logits (n, T, V)."""
    def __init__(self, rnn, vocab_size, embed_size=EMBED, hidden_size=HIDDEN):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = rnn
        self.out = nn.Linear(hidden_size, vocab_size)

    def forward(self, X):
        outputs, _ = self.rnn(self.embedding(X.T))     # (T, n, H); an LSTM's state is a pair, ignored here
        return self.out(outputs).swapaxes(0, 1)


def lm_table(runs):
    if TAG:
        print(TAG)
    print(f"{'model':12s} {'parameters':>10s} {'steps':>6s} {'val perplexity':>14s} {'tokens/s':>9s} {'seconds':>8s}")
    print(f"{'unigram':12s} {V:10,d} {'-':>6s} {unigram_ppl:14.1f}")
    for name, (m, h) in runs.items():
        print(f"{name:12s} {n_params(m):10,d} {LM_STEPS:6d} {h['final_val_ppl']:14.1f} {h['tokens_per_s']:9,.0f} {h['seconds']:8.0f}")


# Overfit one batch (given): your RNN + your clipping must memorize 16 windows of 32 words.
torch.manual_seed(SEED)
Xb, Yb = next(iter(lm.get_dataloader(train=True)))
Xb, Yb = Xb[:16, :32], Yb[:16, :32]
overfit_one_batch("RNN language model", RNNLM(MyRNN(64, 128), V, 64, 128),
                  lambda m: F.cross_entropy(m(Xb).reshape(-1, V), Yb.reshape(-1)), clip_fn=clip_gradients)


In [ ]:
P3 = {}
torch.manual_seed(SEED)
model = RNNLM(MyRNN(EMBED, HIDDEN), V)
P3["your RNN"] = (model, cached("p3_rnn_yours", model, lambda: lm_common.train_lm(
    model, lm, TRAIN_DEVICE, LM_STEPS, clip_fn=clip_gradients, progress=progress, desc="your RNN")))
torch.manual_seed(SEED)
model = RNNLM(nn.RNN(EMBED, HIDDEN), V)
P3["nn.RNN"] = (model, cached("p3_rnn_nn", model, lambda: lm_common.train_lm(
    model, lm, TRAIN_DEVICE, LM_STEPS, progress=progress, desc="nn.RNN")))
lm_table(P3)


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): validation perplexity unigram 303.5 (exact), both RNNs about 22–25.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** Compare both RNNs with the unigram baseline, and with each other (perplexity
and speed).

*How to think about it.* Translate perplexity into "the model hesitates between about N words". What
does the RNN know that the unigram model can't? For the speed: what work does each time step do, and does
the Python loop add much to it?

*Your answer here.*


## (B) GRU and LSTM [15 points]

Gated cells (d2l 10.1–10.2) add learned **gates**: numbers between 0 and 1 that decide, for every hidden
unit at every step, how much of the old state to keep and how much new input to write. The LSTM keeps a
separate *memory cell* $C_t = F_t\odot C_{t-1} + I_t\odot\tilde C_t$ that is updated mostly by addition;
the GRU merges this into its hidden state with an update gate. One-line swaps in `RNNLM`. (Same width:
the GRU has 3× and the LSTM 4× the recurrent weights of the plain RNN.)


In [ ]:
torch.manual_seed(SEED)
gru_layer = ...  ### TODO_STUDENT ### a one-layer GRU, input EMBED, hidden HIDDEN (time-major, the default)
model = RNNLM(gru_layer, V)
P3["nn.GRU"] = (model, cached("p3_gru", model, lambda: lm_common.train_lm(
    model, lm, TRAIN_DEVICE, LM_STEPS, progress=progress, desc="nn.GRU")))
torch.manual_seed(SEED)
lstm_layer = ...  ### TODO_STUDENT ### a one-layer LSTM, input EMBED, hidden HIDDEN
model = RNNLM(lstm_layer, V)
P3["nn.LSTM"] = (model, cached("p3_lstm", model, lambda: lm_common.train_lm(
    model, lm, TRAIN_DEVICE, LM_STEPS, progress=progress, desc="nn.LSTM")))
lm_table(P3)
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, (m, h) in P3.items():
    ax.plot(h["step"], h["val_ppl"], marker="o", ms=3, label=name)
ax.axhline(unigram_ppl, color="gray", ls=":", label="unigram")
ax.set_yscale("log"); ax.set_xlabel("training step"); ax.set_ylabel("validation perplexity"); ax.legend(); ax.grid(alpha=0.3)
ax.set_title(TAG + "validation perplexity")
plt.show()
SUMMARY["P3 ppl"] = {k: round(h["final_val_ppl"], 1) for k, (m, h) in P3.items()}
SUMMARY["P3 tokens/s"] = {k: round(h["tokens_per_s"]) for k, (m, h) in P3.items()}


In [ ]:
PROMPT = "once upon a time, there was a little girl named lily."
LM_SAMPLES = {}                    # saved for HW7 Problem 5 (same prompt, same sampling)
for name in ["nn.RNN", "nn.GRU", "nn.LSTM"]:
    for temperature in [0.5, 1.0]:
        story = lm_common.generate(P3[name][0], lm.vocab, PROMPT, TRAIN_DEVICE, n_words=70, temperature=temperature)
        LM_SAMPLES[f"{name}, temperature {temperature}"] = story
        print(f"--- {TAG}{name}, temperature {temperature} ---\n{story}\n")


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): perplexity GRU about 16.5–18.5, LSTM about 17.5–19.5 (both clearly below the RNN).
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**Generated stories.** The next word is *sampled* from softmax(logits / temperature). Temperature
below 1 sharpens the distribution (safer, more repetitive text), above 1 flattens it (more varied, more
mistakes). Compare the models' stories and the two temperatures.

*How to think about it.* Read each story for two things: is each sentence grammatical, and which
characters and scenes carry over from sentence to sentence (and where does the story contradict itself)?
Which of those needs memory over many words? One sample per model can be noisy: if you see no clear
winner, say so and lean on the perplexity table. Then compare the temperatures for repetition vs mistakes.

*Your answer here.*


**Side question.** TinyStories was built so that about 5,000 words cover 99.6% of it. What would go
wrong with whole words as tokens on real text, and what do real language models use instead?

*How to think about it.* How many distinct words (names, numbers, typos, word forms) are in all of
Wikipedia? What would that do to the embedding table and the output layer, and to a word the model never
saw? Could tokens be smaller than words but larger than letters?

*Your answer here.*


## (C) How far back do gradients reach? [15 points]

The given experiment measures how much the prediction at the **last** position of a 96-word window
depends on each earlier word: the size of the gradient of that prediction's loss with respect to the
input embedding of the word $d$ steps back ($d$ = 0…79), averaged over 64 validation windows, for the
trained RNN, GRU and LSTM. (The first words of each window are left out: they are read from an all-zero
state, a special case.) Run it and describe what the plot shows; Problem 5 then asks *why*.


In [ ]:
def gradient_vs_distance(model, n_windows=64, length=96, max_distance=80):
    """||d loss(last position) / d embedding(t)||, averaged over windows, as a function of the distance T-1-t."""
    g = torch.Generator().manual_seed(123)
    starts = torch.randint(len(lm.val_ids) - length - 1, (n_windows,), generator=g).tolist()
    win = torch.stack([lm.val_ids[s:s + length + 1] for s in starts]).to(TRAIN_DEVICE)
    X, Y = win[:, :-1], win[:, 1:]
    emb = model.embedding(X.T).detach().requires_grad_(True)       # (T, n, E): the inputs we differentiate by
    outputs, _ = model.rnn(emb)
    loss = F.cross_entropy(model.out(outputs[-1]), Y[:, -1])        # only the LAST position's prediction
    loss.backward()
    model.zero_grad()
    return emb.grad.norm(dim=2).mean(dim=1).flip(0)[:max_distance].cpu()   # index d = distance back from the last word


fig, ax = plt.subplots(figsize=(6.5, 3.8))
grad_dist = {}
for name in ["nn.RNN", "nn.GRU", "nn.LSTM"]:
    gd = gradient_vs_distance(P3[name][0])
    grad_dist[name] = gd / gd[0]
    ax.semilogy(grad_dist[name], label=name)
ax.set_xlabel("distance d back from the predicted position (words)")
ax.set_ylabel("gradient size (relative to d = 0)")
ax.set_title(TAG + "how much the last prediction depends on the word d steps back")
ax.grid(alpha=0.3, which="both"); ax.legend()
plt.show()
for d in [1, 5, 10, 20, 40, 60, 79]:
    print(f"{TAG}d = {d:2d}: " + "   ".join(f"{k} {v[d]:.1e}" for k, v in grad_dist.items()))
SUMMARY["P3 grad at d=20"] = {k: float(f"{v[20]:.2g}") for k, v in grad_dist.items()}


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): relative gradient at d = 79: RNN about 10⁻⁵–10⁻⁴, GRU and LSTM about 10⁻³–5·10⁻² (100× or more above the RNN); similar curves up to d ≈ 10.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**Explain the plot.**

*How to think about it.* What does a straight line on a log-scale plot mean? Read the printed values at
d = 10, 20, 40, 79: at which distances are the RNN and the gated models clearly apart? If a word's
gradient is 10⁻⁵ of the last word's, can training teach the model to use that word? Connect the curves
to the perplexities in (B).

*Your answer here.*


---
# PROBLEM 4 — Sequence-to-Sequence Translation, Spanish → English [50 points]

An **encoder–decoder** (d2l 10.6–10.7): the *encoder* RNN reads the Spanish sentence and its final hidden
state summarizes it in one vector, the **context**. The *decoder* is Problem 3's language model, now
*conditioned* on that context: it predicts the English sentence word by word, reading the context at
every step.

*Library calls:* `torch.nn.GRU`, `torch.nn.CrossEntropyLoss` (with `ignore_index`); the d2l book's
`d2l.Seq2Seq` as the package baseline.

**Data** (given, `seq2seq_common.TatoebaEsEn`, a subclass of the book's `d2l.MTFraEng`). Tatoeba's
Spanish–English pairs, lower-cased, punctuation split off (including ¿ and ¡), both sides at most 12
tokens. The splits are by *Spanish sentence*: every test sentence is new to the model, and all its English
translations count as references. The batch format is the book's: `src` = Spanish IDs + `<eos>` + padding,
`tgt_input` = `<bos>` + English IDs, `label` = English IDs + `<eos>` (the decoder's input shifted by one,
as in Problem 3), and `src_valid_len` = the number of real (non-padding) source tokens.


In [ ]:
mt = seq2seq_common.TatoebaEsEn(DATA, batch_size=128, max_len=12, num_train=C["mt_train"])   # same test set in both modes
print(mt.describe())
src, tgt_in, src_len, label = next(iter(mt.get_dataloader(train=True)))
print("\nbatch shapes:", tuple(src.shape), tuple(tgt_in.shape), tuple(src_len.shape), tuple(label.shape))
for name, row, vocab in [("src", src[0], mt.src_vocab), ("tgt_input", tgt_in[0], mt.tgt_vocab), ("label", label[0], mt.tgt_vocab)]:
    print(f"{name:10s}", " ".join(vocab.to_tokens(row.tolist())))
print("src_valid_len[0] =", src_len[0].item())
PAD = mt.tgt_vocab["<pad>"]
MT_STEPS = C["mt_steps"]


## (A) Encoder [10 points]

Embedding + GRU (2 layers, time-major as in the book: the batch is transposed to (steps, n)). The given
`packed_rnn` runs the GRU **only over the real tokens** of each sentence (PyTorch's packed sequences), so
the final state is taken right after each sentence's own `<eos>`, not after the padding. Returns the
outputs at every position (steps, n, H) and the final state (layers, n, H).


In [ ]:
class Encoder(d2l.Encoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.0):
        super().__init__()
        self.embedding = ...  ### TODO_STUDENT ### one learned vector per Spanish token ID
        self.rnn = ...  ### TODO_STUDENT ### a num_layers GRU (time-major), with dropout between layers

    def forward(self, X, valid_len):
        embs = ...  ### TODO_STUDENT ### (n, steps) -> embeddings of shape (steps, n, embed)
        outputs, state = seq2seq_common.packed_rnn(self.rnn, embs, valid_len)
        return outputs, state


EMB, HID, LAYERS, DROPOUT = C["mt_embed"], C["mt_hidden"], C["mt_layers"], 0.2     # full: 256, 512, 2
enc = Encoder(len(mt.src_vocab), EMB, HID, LAYERS, DROPOUT)
outputs, state = enc(src, src_len)
print("encoder outputs", tuple(outputs.shape), "| final state", tuple(state.shape))
assert tuple(outputs.shape) == (mt.num_steps, 128, HID) and tuple(state.shape) == (LAYERS, 128, HID)
print("CHECK passed: encoder shapes (steps, batch, hidden) and (layers, batch, hidden)")
enc.eval()
o_full, s_full = enc(src[:1], src_len[:1])
o_trim, s_trim = enc(src[:1, :src_len[0]], src_len[:1])
check("the final state ignores the padding", s_full, s_trim, tol=1e-5)


## (B) Decoder, trained with teacher forcing [15 points]

The decoder's state is a pair: the **context** (the encoder's final top-layer state, fixed for the whole
sentence) and its own GRU state (initialized with the encoder's final state). At every step its GRU reads
the embedding of the previous English word **concatenated with the context**; a linear layer turns the
GRU's output into scores over the English vocabulary.

**Teacher forcing:** in training, the decoder reads the *true* previous word (`tgt_input`), not its own
guess, so all positions are computed in one call, like Problem 3's language model.

**Masked loss:** a batch is padded to 13 positions, but a 5-word sentence has only 6 real targets
(5 words + `<eos>`). Positions whose label is `<pad>` must not count.


In [ ]:
class Decoder(d2l.Decoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0.0):
        super().__init__()
        self.embedding = ...  ### TODO_STUDENT ### one learned vector per English token ID
        self.rnn = ...  ### TODO_STUDENT ### a GRU whose input is word embedding + context
        self.dense = ...  ### TODO_STUDENT ### scores over the English vocabulary

    def init_state(self, enc_all_outputs, *args):
        outputs, state = enc_all_outputs
        return [state[-1], state]          # [context = encoder's final top-layer state, initial GRU state]

    def forward(self, X, state):
        context, hidden = state
        embs = ...  ### TODO_STUDENT ### (n, steps) -> (steps, n, embed)
        ctx = ...  ### TODO_STUDENT ### the same context at every step: (steps, n, hidden)
        outputs, hidden = ...  ### TODO_STUDENT ### run the GRU on [embedding, context]
        logits = ...  ### TODO_STUDENT ### scores, back to batch-first: (n, steps, vocab)
        return logits, [context, hidden]


def masked_ce(logits, labels, pad):
    """Cross-entropy averaged over the positions whose label is not padding."""
    losses = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1), reduction="none")
    mask = ...  ### TODO_STUDENT ### 1.0 at real target tokens, 0.0 at padding
    return ...  ### TODO_STUDENT ### the average over the real tokens only


def make_translator(data):
    model = d2l.EncoderDecoder(Encoder(len(data.src_vocab), EMB, HID, LAYERS, DROPOUT),
                               Decoder(len(data.tgt_vocab), EMB, HID, LAYERS, DROPOUT))
    model.apply(d2l.init_seq2seq)          # the book's initialization (Xavier), as for the baseline
    return model


# Checkpoint 1: the masked loss vs nn.CrossEntropyLoss(ignore_index=PAD), on a batch with padding.
lg = torch.randn(4, 13, len(mt.tgt_vocab), dtype=torch.float64)
check("masked_ce vs CrossEntropyLoss(ignore_index)", masked_ce(lg, label[:4], PAD),
      nn.CrossEntropyLoss(ignore_index=PAD)(lg.reshape(-1, lg.shape[-1]), label[:4].reshape(-1)), tol=1e-12)
# Checkpoint 2: decoding the whole target at once (teacher forcing) = one word at a time.
model = make_translator(mt).double().eval()
full = model(src[:8], tgt_in[:8], src_len[:8])
state = model.decoder.init_state(model.encoder(src[:8], src_len[:8]), src_len[:8])
steps = []
for t in range(tgt_in.shape[1]):
    lg_t, state = model.decoder(tgt_in[:8, t:t + 1], state)
    steps.append(lg_t)
check("teacher-forced scores = step-by-step scores", full, torch.cat(steps, dim=1), tol=1e-10)
# Checkpoint 3: the decoder must READ the context. Change only the context (same words, same GRU state).
state = model.decoder.init_state(model.encoder(src[:8], src_len[:8]), src_len[:8])
other = [torch.randn_like(state[0]), state[1]]
change = (model.decoder(tgt_in[:8], state)[0] - model.decoder(tgt_in[:8], other)[0]).abs().amax(dim=(0, 2))
assert change.min() > 1e-6, "changing the context does not change the scores at every step: does the GRU read it?"
print(f"CHECK passed: the scores depend on the context at every step (smallest change {change.min():.2g})")
print("logits shape", tuple(full.shape), "| parameters:", f"{n_params(model):,}")
# Overfit one batch (given): a small translator built from your classes must memorize 16 sentence pairs.
torch.manual_seed(SEED)
b_src, b_tgt, b_len, b_lab = (a[:16] for a in (src, tgt_in, src_len, label))
tiny = d2l.EncoderDecoder(Encoder(len(mt.src_vocab), 32, 64, 2), Decoder(len(mt.tgt_vocab), 32, 64, 2))
overfit_one_batch("seq2seq translator", tiny, lambda m: masked_ce(m(b_src, b_tgt, b_len), b_lab, PAD))


**Train it** (given: `seq2seq_common.train_seq2seq`; AdamW, learning rate $10^{-3}$, batch 128,
clipping at 1, random batches). The book's package baseline is trained right after (d), with the same
loop, data, sizes and number of steps.


In [ ]:
torch.manual_seed(SEED)
translator = make_translator(mt)
mt_hist = cached("p4_translator", translator, lambda: seq2seq_common.train_seq2seq(
    translator, mt, TRAIN_DEVICE, MT_STEPS, loss_fn=lambda logits, labels: masked_ce(logits, labels, PAD),
    progress=progress, desc="GRU seq2seq"))
print(f"{TAG}{MT_STEPS} steps ({MT_STEPS * mt.batch_size / len(mt.arrays[0]):.1f} epochs) in {mt_hist['seconds']:.0f} s; "
      f"validation loss {mt_hist['val_loss'][-1]:.3f}")
plt.figure(figsize=(5.5, 3.2))
plt.plot(mt_hist["step"], mt_hist["train_loss"], label="training (running average)")
plt.plot(mt_hist["step"], mt_hist["val_loss"], label="validation")
plt.xlabel("step"); plt.ylabel("masked cross-entropy"); plt.legend(); plt.grid(alpha=0.3)
plt.show()


## (C) Greedy decoding [10 points]

At test time there is no true previous word: the decoder starts from `<bos>`, picks its most likely word,
feeds that word back in as the next input, and repeats, until it produces `<eos>` (or a length cap).
Write the loop for a whole batch of sentences at once. The checkpoint compares it with the book's
`predict_step`, which does the same thing.


In [ ]:
@torch.no_grad()
def greedy_translate(model, data, sentences, max_steps=20):
    """Spanish token lists -> English token lists (without <eos>), greedy, one batch."""
    model.eval()
    src, src_len = (a.to(TRAIN_DEVICE) for a in data.src_batch(sentences))
    state = ...  ### TODO_STUDENT ### encode, then the decoder's initial state
    Y = torch.full((len(src), 1), data.tgt_vocab["<bos>"], device=TRAIN_DEVICE)    # every translation starts with <bos>
    outputs = []
    for _ in range(max_steps):
        logits, state = ...  ### TODO_STUDENT ### one decoder step on the previous words Y (n, 1)
        Y = ...  ### TODO_STUDENT ### the most likely next word of each sentence, shape (n, 1)
        outputs.append(Y)
        if (torch.cat(outputs, 1) == data.tgt_vocab["<eos>"]).any(dim=1).all():
            break                          # every sentence has produced <eos>
    model.train()
    return [seq2seq_common.ids_to_tokens(row, data.tgt_vocab) for row in torch.cat(outputs, 1).tolist()]


examples = ["¿ dónde está la estación ?", "tengo hambre .", "mi hermano trabaja en un hospital .",
            "no sé qué hacer .", "ella me dio un libro muy interesante ."]
for es, en in zip(examples, greedy_translate(translator, mt, [s.split() for s in examples])):
    print(f"{TAG}{es:40s} -> {' '.join(en)}")

# Checkpoint: identical to the book's predict_step on the first 200 test sentences.
sents = mt.test_src[:200]
s_arr, s_len = mt.src_batch(sents)
batch = (s_arr, torch.full((len(sents), 1), mt.tgt_vocab["<bos>"]), s_len, torch.zeros(1))   # (labels unused)
translator.eval()
book_ids, _ = translator.predict_step(batch, TRAIN_DEVICE, 20)
translator.train()
book = [seq2seq_common.ids_to_tokens(row, mt.tgt_vocab) for row in book_ids.tolist()]
assert greedy_translate(translator, mt, sents) == book, "greedy_translate differs from d2l's predict_step"
print("CHECK passed: greedy_translate = d2l predict_step on 200 test sentences")


## (D) Evaluate and save [15 points]

**BLEU** (Papineni et al., 2002) compares a translation with its references by counting shared n-grams
(n = 1…4). Each n-gram of the translation counts at most as often as it appears in any **one**
reference (*clipped* counts: "the the the the" can't score four matches against "the cat"). The four
precisions are combined by a geometric mean, and a **brevity penalty** punishes translations shorter
than their references. We use **corpus BLEU-4**: counts are summed over the whole test set first
(`seq2seq_common.corpus_bleu`, the same formula as the standard tool sacrebleu; the book's `d2l.bleu` is a
different, per-sentence variant whose numbers are not comparable). Write the clipped count — every BLEU
number below is computed with **your** `clipped_matches` (`clip_fn=`) — then BLEU by source-sentence
length (in tokens, punctuation included).


In [ ]:
def clipped_matches(hyp, refs, n):
    """Number of n-grams of hyp that also occur in a reference, each counted at most as often as in any ONE reference."""
    hyp_counts = seq2seq_common.ngrams(hyp, n)                  # Counter: n-gram (tuple) -> count in hyp
    most = collections.Counter()
    for ref in refs:
        for gram, c in seq2seq_common.ngrams(ref, n).items():
            most[gram] = ...  ### TODO_STUDENT ### the most times this n-gram occurs in any single reference
    return ...  ### TODO_STUDENT ### each hyp n-gram counts at most that many times


check("clipped unigram matches: 'the' x7 vs 'the cat is on the mat'",
      clipped_matches(["the"] * 7, [["the", "cat", "is", "on", "the", "mat"]], 1), 2)
check("two references: 'the the the cat' scores 3 ('the' counts 2, its max in ONE reference; not 3, the sum)",
      clipped_matches("the the the cat".split(), [["the", "cat"], ["the", "the", "dog"]], 1), 3)
toy = seq2seq_common.corpus_bleu([["the", "cat", "sat", "on", "mat"]], [[["the", "cat", "sat", "on", "the", "mat"]]],
                                 clip_fn=clipped_matches)
check("BLEU-4 of the lecture's toy pair (p = 1, 3/4, 2/3, 1/2; BP = e^-0.2)", toy, 57.893, tol=1e-3)


def bleu_by_length(hyps, refs, srcs, buckets=seq2seq_common.BUCKETS):
    out = {}
    for lo, hi in buckets:
        keep = [i for i, s in enumerate(srcs) if lo <= len(s) <= hi]  # (given) test sentences whose Spanish length is in [lo, hi]
        out[f"{lo}-{hi}"] = seq2seq_common.corpus_bleu([hyps[i] for i in keep], [refs[i] for i in keep], clip_fn=clipped_matches)  # (given) BLEU on that subset (with your clipped_matches)
    return out


# Checkpoint: three toy test sentences of 2, 6 and 10 Spanish tokens; only the first and last are translated right.
good = "the cat sat on the mat".split()
toy_by_len = bleu_by_length([good, "a b c d".split(), good], [[good], [good], [good]], [["x"] * 2, ["x"] * 6, ["x"] * 10])
check("bleu_by_length on 3 toy sentences (buckets by SOURCE length)", list(toy_by_len.values()), [100, 0, 0, 100], tol=1e-9)


t0 = time.time()
hyps = sum((greedy_translate(translator, mt, mt.test_src[i:i + 500]) for i in range(0, len(mt.test_src), 500)), [])
bleu = seq2seq_common.corpus_bleu(hyps, mt.test_refs, clip_fn=clipped_matches)
by_len = bleu_by_length(hyps, mt.test_refs, mt.test_src)
n_in_bucket = {k: sum(lo <= len(s) <= hi for s in mt.test_src) for k, (lo, hi) in zip(by_len, seq2seq_common.BUCKETS)}
print(f"translated {len(hyps):,} test sentences in {time.time() - t0:.1f} s")
print(f"{TAG}test BLEU-4: {bleu:.1f}")
print(TAG + "by Spanish length:", "  ".join(f"{k} tokens: {v:.1f} ({n_in_bucket[k]} sentences)" for k, v in by_len.items()))
SUMMARY["P4 BLEU (yours)"] = (round(bleu, 1), {k: round(v, 1) for k, v in by_len.items()})


In [ ]:
def show_examples(hyps, title, idx):
    print(title)
    for i in idx:
        print(f"  ES  {' '.join(mt.test_src[i])}\n  OUT {' '.join(hyps[i])}\n  REF {' | '.join(' '.join(r) for r in mt.test_refs[i])}\n")


exact = [i for i, h in enumerate(hyps) if any(h == r for r in mt.test_refs[i])]
print(f"{TAG}{len(exact)} of {len(hyps)} test translations ({len(exact) / len(hyps):.0%}) match a reference exactly\n")
show_examples(hyps, "GOOD (exact matches):", exact[:4])
overlap = [len(set(h) & set(w for r in mt.test_refs[i] for w in r)) / max(len(h), 1) for i, h in enumerate(hyps)]
long_bad = sorted([i for i, s in enumerate(mt.test_src) if len(s) >= 9], key=lambda i: overlap[i])[:4]
show_examples(hyps, "BAD (long sentences with the least word overlap):", long_bad)


**The book's package baseline** (given): `d2l.Seq2Seq` with the package's `d2l.Seq2SeqEncoder` and
the book's decoder (copied into `seq2seq_common`, since the package doesn't ship it), same sizes, data,
loop and number of steps. One real difference from your model: the book's encoder reads the padding too,
and its context is its output at the *last padded position*, not right after the sentence's `<eos>`.


In [ ]:
torch.manual_seed(SEED)
book_model = seq2seq_common.book_seq2seq(mt, EMB, HID, LAYERS, DROPOUT)
book_hist = cached("p4_book", book_model, lambda: seq2seq_common.train_seq2seq(
    book_model, mt, TRAIN_DEVICE, MT_STEPS, loss_fn=book_model.loss, progress=progress, desc="d2l.Seq2Seq"))
book_hyps = sum((greedy_translate(book_model, mt, mt.test_src[i:i + 500]) for i in range(0, len(mt.test_src), 500)), [])
book_bleu = seq2seq_common.corpus_bleu(book_hyps, mt.test_refs, clip_fn=clipped_matches)
book_by_len = bleu_by_length(book_hyps, mt.test_refs, mt.test_src)
print(f"{TAG}\n{'model':28s} {'params':>11s} {'seconds':>8s} {'BLEU':>6s}   " + "  ".join(f"{k:>5s}" for k in by_len))
for name, m, h, b, bl in [("yours (packed encoder)", translator, mt_hist, bleu, by_len),
                          ("d2l.Seq2Seq (book)", book_model, book_hist, book_bleu, book_by_len)]:
    print(f"{name:28s} {n_params(m):11,d} {h['seconds']:8.0f} {b:6.1f}   " + "  ".join(f"{v:5.1f}" for v in bl.values()))
x = np.arange(len(by_len))
plt.figure(figsize=(5.5, 3.2))
plt.bar(x - 0.2, list(by_len.values()), 0.4, label="yours")
plt.bar(x + 0.2, list(book_by_len.values()), 0.4, label="d2l.Seq2Seq")
plt.xticks(x, [f"{k} tokens" for k in by_len]); plt.ylabel("BLEU-4"); plt.title(TAG + "BLEU by Spanish sentence length")
plt.legend(); plt.grid(alpha=0.3, axis="y"); plt.show()
SUMMARY["P4 BLEU (book)"] = (round(book_bleu, 1), {k: round(v, 1) for k, v in book_by_len.items()})


**Save for HW7** (given). HW7's attention models and Transformers are compared with this model on the
same test sentences. `results_hw6.json` holds the BLEU numbers and every test translation;
`hw6_seq2seq.pt` the weights, sizes and vocabularies. The file also keeps Problem 3's language-model
perplexities and stories, which HW7's GPT is compared with. **HW7 needs these files from your own FULL
run**; a dev run writes `results_hw6_dev.json` and `hw6_seq2seq_dev.pt` instead (HW7 refuses them).


In [ ]:
RESULTS_FILE = "results_hw6.json" if MODE == "full" else "results_hw6_dev.json"
CKPT_FILE = "hw6_seq2seq.pt" if MODE == "full" else "hw6_seq2seq_dev.pt"
config = {"embed_size": EMB, "num_hiddens": HID, "num_layers": LAYERS, "dropout": DROPOUT}
lm_results = {                     # Problem 3's language models, for HW7 Problem 5
    "data": {k: getattr(lm, k) for k in ("batch_size", "num_steps", "min_freq", "val_fraction")} | {"vocab_size": V},
    "unigram_ppl": round(unigram_ppl, 2),
    "models": {name: {"val_ppl": round(h["final_val_ppl"], 2), "params": n_params(m), "steps": LM_STEPS,
                      "tokens_per_s": round(h["tokens_per_s"]), "seconds": round(h["seconds"], 1)}
               for name, (m, h) in P3.items()},
    "samples": {"prompt": PROMPT, "n_words": 70, "seed": 0, "texts": LM_SAMPLES},
}
res = seq2seq_common.save_results(RESULTS_FILE, "GRU seq2seq (HW6 P4)", mt, hyps, mt_hist,
                                  clip_fn=clipped_matches, bleu_by_len=by_len, extra={"mode": MODE, "config": config, "params": n_params(translator),
                                         "book_baseline_bleu": round(book_bleu, 2),
                                         "book_baseline_bleu_by_src_len": {k: round(v, 2) for k, v in book_by_len.items()},
                                         "lm": lm_results})
seq2seq_common.save_checkpoint(CKPT_FILE, translator, config, mt)
print(f"{TAG}saved {RESULTS_FILE} (mode {MODE!r}): BLEU", res["bleu"], res["bleu_by_src_len"])
print("  and the language models' perplexities:", {k: v["val_ppl"] for k, v in res["lm"]["models"].items()})
print(f"saved {CKPT_FILE} ({os.path.getsize(CKPT_FILE) / 1e6:.0f} MB)")
assert seq2seq_common.load_results(RESULTS_FILE)["test_hyp"][0] == " ".join(hyps[0])


> **Expected in a full run** (`MODE = "full"`; from the solution's run, with margins): BLEU-4 about 25–30; by Spanish length the 9–12-token bucket about 15–20 BLEU below the 5–6-token one; book baseline about 0–3 BLEU below yours.
>
> In `dev` mode the numbers mean nothing: dev only shows that your code runs.

**What do you see?** How good are the translations, how does quality depend on sentence length, and
how does your model compare with the book's?

*How to think about it.* Use the BLEU table and the good/bad examples. Where in the long sentences do the
mistakes appear: at the start, near the end, everywhere? Everything the decoder knows about the Spanish sentence passes through one vector of 512
numbers. For the book's model: what does its encoder read after a short sentence's `<eos>`?

*Your answer here.*


---
# PROBLEM 5 (THEORY) — Why an LSTM Reaches Further Back Than a Plain RNN [25 points]

Problem 3(C) described *what* the curves show; here explain *why*. Using your Problem 3(C) plot: why does
the plain RNN's gradient shrink with distance, and what in the LSTM changes that? Why does gradient
clipping fix exploding gradients but not vanishing ones?


**Your answer.**

*How to think about it.* Write out what happens to the gradient when you go one step back through
$H_t=\tanh(X_tW_{xh}+H_{t-1}W_{hh}+b_h)$: what gets multiplied in? Then do it $d$ times; what does
$0.9^{d}$ look like on a log plot, and what about $1.1^{d}$? For the LSTM, look at
$C_t=F_t\odot C_{t-1}+I_t\odot\tilde C_t$: what is the factor per step along $C$, and who controls it? For
clipping: what is wrong with an exploding gradient — its direction or its size? And with a vanishing
one?

*Your answer here.*


---
# PROBLEM 6 (THEORY) — What Convolution Is (and Isn't) Robust To [25 points]

A CNN trained on centred, upright objects fails on objects moved to a corner or rotated. Which changes
does convolution + pooling handle on its own, which does it not, and which of these did your Problem 2(B)
augmentation teach it?


**Your answer.**

*How to think about it.* Take one kernel that detects a vertical edge. What happens to its output map if
the image moves 3 pixels right? If the image is rotated by 90°? Then add 2×2 max pooling: when does a
1-pixel shift leave the pooled output unchanged, and when does it change it (where are the window
boundaries)? Use Problem 1(C)'s shift test as evidence. Finally, list exactly what
`augment` did to the training images — and what it never did.

*Your answer here.*


---
# OPTIONAL PROBLEMS [no credit]

Problems 7 (plain vs residual networks at 20+ layers), 9 (your Problem 3 RNN inside the Problem 4
translator) and 10 (an LSTM translator) are described in the HTML only. Swapping `nn.GRU` for `nn.LSTM`
in `Encoder`/`Decoder` is a few-line change (the LSTM's state is a pair `(h, c)`). Your `MyRNN` takes
more work: `packed_rnn` hands the encoder a `PackedSequence`, which `MyRNN` can't read, and the
translator has 2 layers — run your encoder without packing (it then reads the padding, like the book's),
stack two `MyRNN`s (or wrap them in a small 2-layer module with `nn.GRU`'s interface), and expect slow
training.

## PROBLEM 8 — An LSTM cell from scratch

The LSTM equations in PyTorch's convention (d2l 10.1 uses the same gates), for input $x$ (n × E), state
$h$ (n × H), cell $c$ (n × H):
$I=\sigma(xW_{xi}+hW_{hi}+b_i)$, $F=\sigma(xW_{xf}+hW_{hf}+b_f)$, $\tilde C=\tanh(xW_{xc}+hW_{hc}+b_c)$,
$O=\sigma(xW_{xo}+hW_{ho}+b_o)$; then $c'=F\odot c+I\odot\tilde C$ and $h'=O\odot\tanh(c')$. `nn.LSTM` stores
the four gates' weights stacked in the order (i, f, g = $\tilde C$, o).


In [ ]:
def lstm_step(x, h, c, W_x, W_h, b):
    """One LSTM step. W_x (E, 4H), W_h (H, 4H), b (4H,): the four gates side by side in the order i, f, g, o."""
    z = x @ W_x + h @ W_h + b
    i, f, g, o = z.chunk(4, dim=1)
    I, Fg, O = ...  ### TODO_STUDENT ### input, forget and output gates
    C_tilde = ...  ### TODO_STUDENT ### the candidate cell
    c_new = ...  ### TODO_STUDENT ### keep part of the old cell, add part of the candidate
    h_new = ...  ### TODO_STUDENT ### the new hidden state
    return h_new, c_new


if RUN_OPTIONAL:
    ref = nn.LSTM(5, 7).double()
    W_x, W_h = ref.weight_ih_l0.T, ref.weight_hh_l0.T
    b = ref.bias_ih_l0 + ref.bias_hh_l0
    Xs = torch.randn(9, 3, 5, dtype=torch.float64)
    h, c = torch.zeros(3, 7, dtype=torch.float64), torch.zeros(3, 7, dtype=torch.float64)
    outs = []
    for x in Xs:
        h, c = lstm_step(x, h, c, W_x, W_h, b)
        outs.append(h)
    out_r, (h_r, c_r) = ref(Xs)
    check("your LSTM outputs vs nn.LSTM (9 steps)", torch.stack(outs), out_r, tol=1e-12)
    check("your LSTM final cell vs nn.LSTM", c, c_r[0], tol=1e-12)


---
## PROBLEM 11 (optional, no points) — Why dev mode looks the way it does

Look at the two columns the *Run settings* cell prints. Short answers.


**(a)** Dev trains Problem 2's ResNet, half as wide, for 3 epochs on 2,000 CIFAR images. Why are its
accuracies meaningless, while the checkpoints and checks in the same run are still trustworthy?

*How to think about it.* What does a test accuracy measure, and what does a checkpoint measure? Which of
the two depends on how much data and training the model got?

*Your answer here.*


**(b)** Dev's language model keeps only words seen at least 50 times (full: 10). Why is a dev
perplexity not comparable with a full one, even apart from the smaller model?

*How to think about it.* What happens to all the rare words? Is predicting `<unk>` hard or easy? Compare
the unigram perplexity printed in dev with the full one.

*Your answer here.*


**(c)** Dev's translator still sees all four length buckets (the test set is the same), but it can't
show the long-sentence BLEU drop. Why? And what would make dev *too* small to be useful?

*How to think about it.* What has to be true of a model before "worse on long sentences" can be seen at
all? For the second part: list the code paths dev must still run (padding, several batches, every
bucket, the loss going down...).

*Your answer here.*


---
## Summary of the key numbers (given)


In [ ]:
import json
print(TAG + json.dumps(SUMMARY, indent=1))
